In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/sentinelai', exist_ok=True)
%cd /content/drive/MyDrive/sentinelai

Mounted at /content/drive
/content/drive/MyDrive/sentinelai


### Step 1.1 - Get the AIT Alert Data Set (authors' CSV export, 2,655,821 alerts)

In [2]:
!git clone --depth 1 https://github.com/ait-aecid/alert-data-set.git
!unzip -oq alert-data-set/alerts_csv/alerts_csv.zip -d data
!ls -lh data/alerts_csv

fatal: destination path 'alert-data-set' already exists and is not an empty directory.
total 252M
-rw------- 1 root root  44M Mar 25  2024 fox_alerts.txt
-rw------- 1 root root  58M Mar 25  2024 harrison_alerts.txt
-rw------- 1 root root 4.4M Mar 25  2024 russellmitchell_alerts.txt
-rw------- 1 root root  13M Mar 25  2024 santos_alerts.txt
-rw------- 1 root root 6.8M Mar 25  2024 shaw_alerts.txt
-rw------- 1 root root 8.7M Mar 25  2024 wardbeck_alerts.txt
-rw------- 1 root root  58M Mar 25  2024 wheeler_alerts.txt
-rw------- 1 root root  60M Mar 25  2024 wilson_alerts.txt


### Step 1.2 - Look at one network (russellmitchell, the smallest)

In [3]:
import pandas as pd
df = pd.read_csv('data/alerts_csv/russellmitchell_alerts.txt')
print(len(df), 'alerts')
print(pd.to_datetime(df.time.min(), unit='s'), '->', pd.to_datetime(df.time.max(), unit='s'))
df.head(10)

45544 alerts
2022-01-21 00:00:01 -> 2022-01-24 23:39:13


,time,name,ip,host,short,time_label,event_label
0,1642723347,Wazuh: ClamAV database update,172.19.130.4,mail,W-Sys-Cav,false_positive,-
1,1642723352,Wazuh: ClamAV database update,172.19.130.4,mail,W-Sys-Cav,false_positive,-
2,1642723357,Wazuh: ClamAV database update,172.19.130.4,mail,W-Sys-Cav,false_positive,-
3,1642723362,Wazuh: ClamAV database update,172.19.130.4,mail,W-Sys-Cav,false_positive,-
4,1642723367,Wazuh: ClamAV database update,172.19.130.4,mail,W-Sys-Cav,false_positive,-
5,1642723368,Wazuh: ClamAV database update,172.19.130.4,mail,W-Sys-Cav,false_positive,-
6,1642723432,Wazuh: ClamAV database update,192.168.231.56,davey_mail,W-Sys-Cav,false_positive,-
7,1642724061,Suricata: Alert - ET POLICY GNU/Linux APT User...,10.143.0.103,internal_share,S-Flw-Apt,false_positive,-
8,1642724061,Wazuh: First time this IDS alert is generated.,10.143.0.103,internal_share,W-All-Ids,false_positive,-
9,1642724061,Suricata: Alert - ET POLICY GNU/Linux APT User...,172.19.128.1,inet-firewall,S-Flw-Apt,false_positive,-


### Step 1.2 - Which tool, which machine, what labels

In [4]:
df['tool'] = df['name'].str.extract(r'^(Wazuh|Suricata|AMiner):', expand=False)
print(df['tool'].value_counts(), '\n')
print(df['host'].value_counts(), '\n')
print(df['time_label'].value_counts())

tool
Wazuh       32302
Suricata     9186
AMiner       4056
Name: count, dtype: int64 

host
intranet_server    11014
inet-firewall      10104
mail                8634
davey_mail          4780
webserver           4318
vpn                 3984
morris_mail         2368
cloud_share          166
inet-dns              84
internal_share        65
monitoring            27
Name: count, dtype: int64 

time_label
false_positive          33529
wpscan                   6359
dirb                     4522
dnsteal                   711
service_scans             367
privilege_escalation       24
cracking                   14
network_scans               8
webshell                    5
reverse_shell               3
service_stop                2
Name: count, dtype: int64


### Steps 1.3-1.6 - The loader: split the answers off, sort by time, save, check totals

In [5]:
import pandas as pd
from pathlib import Path

SCENARIOS = ['fox', 'harrison', 'russellmitchell', 'santos', 'shaw', 'wardbeck', 'wheeler', 'wilson']
OUT = Path('processed')
OUT.mkdir(exist_ok=True)


def load_alert_csv(path, scenario):
    """Reads one <scenario>_alerts.txt and returns two tables: (alerts, labels)."""
    raw = pd.read_csv(path)

    # 1) What the agent is allowed to see
    alerts = pd.DataFrame({
        'alert_id': [f'alert-{scenario}-{i:07d}' for i in range(len(raw))],  # row number in the authors' CSV
        'network': scenario,
        'ts': raw['time'].astype('int64'),                                       # seconds, for gap maths
        'time': pd.to_datetime(raw['time'], unit='s', utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ'),
        'ids_source': raw['name'].str.extract(r'^(Wazuh|Suricata|AMiner):', expand=False).str.lower().fillna('aminer'),
        'signature': raw['name'],
        'short': raw['short'],
        'host': raw['host'],
        'host_ip': raw['ip'],        # the machine that raised the alert (IPs are never dropped)
        'src_ip': None,              # not in the CSV export; the raw Zenodo file has them
        'dst_ip': None,
    })

    # 2) The answers, kept in a separate table (evaluation only)
    labels = pd.DataFrame({
        'alert_id': alerts['alert_id'],
        'eval_phase': raw['time_label'],         # attack phase by time window (noisy)
        'eval_event_label': raw['event_label'],  # exact label where known, else '-'
    })

    # 3) The agent reads alerts in the order they happened
    alerts = alerts.sort_values(['ts', 'alert_id'], kind='stable').reset_index(drop=True)
    return alerts, labels

In [6]:
# Load all 8 networks, save both tables, and check the totals against Zenodo
summary = []
for sc in SCENARIOS:
    alerts, labels = load_alert_csv(f'data/alerts_csv/{sc}_alerts.txt', sc)
    alerts.to_parquet(OUT / f'{sc}_alerts.parquet', index=False)
    labels.to_parquet(OUT / f'{sc}_eval.parquet', index=False)
    c = alerts['ids_source'].value_counts()
    summary.append({'network': sc, 'alerts': len(alerts),
                    'wazuh': int(c.get('wazuh', 0)), 'suricata': int(c.get('suricata', 0)),
                    'aminer': int(c.get('aminer', 0)), 'hosts': alerts['host'].nunique(),
                    'first': alerts['time'].iloc[0], 'last': alerts['time'].iloc[-1],
                    'missing_host_ip': int(alerts['host_ip'].isna().sum())})
    print(f'{sc:16s} {len(alerts):>8,} alerts')

summary = pd.DataFrame(summary)
summary.to_csv(OUT / 'summary.csv', index=False)

total = summary['alerts'].sum()
tools = summary[['wazuh', 'suricata', 'aminer']].sum()
print(f'\nTotal: {total:,}  ->', 'OK' if total == 2_655_821 else 'MISMATCH', '(Zenodo: 2,655,821)')
print('Per tool:', tools.to_dict(), '->',
      'OK' if list(tools) == [2_293_628, 306_635, 55_558] else 'MISMATCH', '(Zenodo: 2,293,628 / 306,635 / 55,558)')
print('Label columns in the agent table:', [c for c in alerts.columns if 'label' in c or c.startswith('eval_')], '(must be empty)')
summary

fox               473,104 alerts
harrison          593,948 alerts
russellmitchell    45,544 alerts
santos            130,779 alerts
shaw               70,782 alerts
wardbeck           91,257 alerts
wheeler           616,161 alerts
wilson            634,246 alerts

Total: 2,655,821  -> OK (Zenodo: 2,655,821)
Per tool: {'wazuh': 2293628, 'suricata': 306635, 'aminer': 55558} -> OK (Zenodo: 2,293,628 / 306,635 / 55,558)
Label columns in the agent table: [] (must be empty)


,network,alerts,wazuh,suricata,aminer,hosts,first,last,missing_host_ip
0,fox,473104,451439,11084,10581,13,2022-01-15T00:00:01Z,2022-01-19T23:55:36Z,0
1,harrison,593948,516370,67384,10194,11,2022-02-04T00:00:01Z,2022-02-08T23:59:46Z,0
2,russellmitchell,45544,32302,9186,4056,11,2022-01-21T00:00:01Z,2022-01-24T23:39:13Z,0
3,santos,130779,82401,44112,4266,11,2022-01-14T00:00:01Z,2022-01-17T23:51:42Z,0
4,shaw,70782,51847,16692,2243,12,2022-01-25T00:00:01Z,2022-01-30T23:31:06Z,0
5,wardbeck,91257,63694,24510,3053,12,2022-01-19T00:00:01Z,2022-01-23T23:59:19Z,0
6,wheeler,616161,535310,68629,12222,13,2022-01-26T00:00:01Z,2022-01-30T23:54:15Z,0
7,wilson,634246,560265,65038,8943,11,2022-02-03T00:00:01Z,2022-02-08T23:50:14Z,0


### Step 1.5 - Attack-phase times (evaluation only; the agent never reads them)

In [7]:
import importlib.util

ATTACK_PHASES = ['network_scans', 'service_scans', 'dirb', 'wpscan', 'webshell', 'cracking',
                 'reverse_shell', 'privilege_escalation', 'service_stop', 'dnsteal']

# Read the phase table from the authors' attacktimes.py (same times as labels.csv on Zenodo)
spec = importlib.util.spec_from_file_location('ait_attacktimes', 'alert-data-set/attacktimes.py')
attacktimes = importlib.util.module_from_spec(spec)
spec.loader.exec_module(attacktimes)

rows = []
for network, windows in attacktimes.phase.items():
    for name, (start, end) in windows.items():
        rows.append({'network': network, 'phase': name,
                     'start': pd.Timestamp(start).tz_convert('UTC'),
                     'end': pd.Timestamp(end).tz_convert('UTC'),
                     'is_attack': name in ATTACK_PHASES})   # false_positive_* windows are not attacks

phases = pd.DataFrame(rows).sort_values(['network', 'start']).reset_index(drop=True)
phases['minutes'] = ((phases['end'] - phases['start']).dt.total_seconds() / 60).round(1)
phases.to_csv(OUT / 'phases.csv', index=False)

print(len(phases), 'phase windows,', int(phases['is_attack'].sum()), 'of them attack steps')
phases[phases['network'] == 'russellmitchell']

95 phase windows, 79 of them attack steps


,network,phase,start,end,is_attack,minutes
24,russellmitchell,false_positive_test,2022-01-23 01:00:00+00:00,2022-01-23 06:00:00+00:00,False,300.0
25,russellmitchell,false_positive_same_day,2022-01-24 00:00:00+00:00,2022-01-25 00:00:00+00:00,False,1440.0
26,russellmitchell,network_scans,2022-01-24 03:01:00+00:00,2022-01-24 03:56:46+00:00,True,55.8
27,russellmitchell,service_scans,2022-01-24 03:56:46+00:00,2022-01-24 03:57:25+00:00,True,0.6
28,russellmitchell,dirb,2022-01-24 03:57:25+00:00,2022-01-24 03:57:48+00:00,True,0.4
29,russellmitchell,wpscan,2022-01-24 03:57:48+00:00,2022-01-24 03:58:19+00:00,True,0.5
30,russellmitchell,webshell,2022-01-24 03:58:19+00:00,2022-01-24 03:59:22+00:00,True,1.0
31,russellmitchell,cracking,2022-01-24 03:59:22+00:00,2022-01-24 04:36:56+00:00,True,37.6
32,russellmitchell,reverse_shell,2022-01-24 04:36:56+00:00,2022-01-24 04:37:39+00:00,True,0.7
33,russellmitchell,privilege_escalation,2022-01-24 04:37:39+00:00,2022-01-24 04:38:13+00:00,True,0.6


In [8]:
# Is the first 24 hours of every network free of attacks?
# (Stage 2 learns 'normal' from that window, so this must be true.)
summary = pd.read_csv(OUT / 'summary.csv')
check = []
for _, r in summary.iterrows():
    att = phases[(phases['network'] == r['network']) & phases['is_attack']]
    first = att.loc[att['start'].idxmin()]
    data_start = pd.Timestamp(r['first'])
    hours = (first['start'] - data_start).total_seconds() / 3600
    check.append({'network': r['network'], 'data_start': data_start,
                  'first_attack_step': first['phase'], 'first_attack_at': first['start'],
                  'hours_before_attack': round(hours, 1), 'first_24h_clean': hours >= 24})

check = pd.DataFrame(check)
print('All 8 networks have an attack-free first 24 hours:', bool(check['first_24h_clean'].all()))
check

All 8 networks have an attack-free first 24 hours: True


,network,data_start,first_attack_step,first_attack_at,hours_before_attack,first_24h_clean
0,fox,2022-01-15 00:00:01+00:00,service_stop,2022-01-17 09:04:46+00:00,57.1,True
1,harrison,2022-02-04 00:00:01+00:00,network_scans,2022-02-08 07:05:00+00:00,103.1,True
2,russellmitchell,2022-01-21 00:00:01+00:00,network_scans,2022-01-24 03:01:00+00:00,75.0,True
3,santos,2022-01-14 00:00:01+00:00,service_stop,2022-01-16 07:16:17+00:00,55.3,True
4,shaw,2022-01-25 00:00:01+00:00,service_stop,2022-01-28 21:08:00+00:00,93.1,True
5,wardbeck,2022-01-19 00:00:01+00:00,service_stop,2022-01-20 22:12:27+00:00,46.2,True
6,wheeler,2022-01-26 00:00:01+00:00,service_stop,2022-01-29 05:27:24+00:00,77.5,True
7,wilson,2022-02-03 00:00:01+00:00,service_stop,2022-02-06 10:47:15+00:00,82.8,True


## Stage 2 - Step 2.1: the linking rules (version 1, in plain words)

**Rule 0 - Never use the answers.** The agent reads only the `alerts` table, never `labels` or `phases`.

**Rule 1 - Merge bursts.** Repeats of the same alert type on the same machine, less than **60 s** apart, become one group; a group lasts at most **10 min**.  
*Why:* dirb made 4,522 alerts in 23 seconds - one event, not 4,522. The cap stops an endless stream (ClamAV every 5 s) from swallowing an attack alert.

**Rule 2 - Filter routine noise without labels.** The first **24 h** of each network are normal (step 1.5: attack-free everywhere). An alert type on a machine is *routine* if it appeared in at least **6 different 10-minute slots** that day. It stays *unusual* if a group is more than **3x** that type's busiest baseline slot (a spike). Routine groups never link.  
*Why:* 74% of alerts are repeating false alarms; attacks often trigger normal alert types, just far more of them.

**Rule 3 - Link only on shared evidence.** A group joins an incident only if it **shares a machine or IP** with it **and** starts **<= 45 min** after that machine/IP was last seen *(tuned from 30 min on 23 Nov: at 30 min privilege escalation split off from the main incident in all 3 tuning networks; 45 min joined it with no loss of purity; longer gaps added nothing but noise and weak links - see 'Tuning (a)')*. **Strong** if the gap is <= 5 min, otherwise **weak**. If several incidents qualify, take the closest. Being close in time alone is **never** enough.

**Rule 4 - Order check** *(implemented 24 Nov, Stage 4, with Omar's Kill Chain stages)*. Each step gets a Kill Chain position from what its alert type may point to (O-1). If a step's stage comes **before** a stage the incident already reached, it is still linked (never dropped), but the link is **weak** and marked `out_of_order`. Unknown stage = no change.

**Rule 5 - Decision (b)** *(implemented 24 Nov, Stage 4)*. A second pass joins a later incident to an earlier one only if they **share a machine or IP**, the gap is **over 45 min but at most 24 h**, and the later incident's earliest stage comes **strictly after** the latest stage the earlier one reached. Such joins are always **weak** and marked `long_gap`. Incidents with no known stage never join. (On the real AIT CSV this makes no joins, because the CSV has no src/dst IPs to share across machines; the scenario shows it working.)

**Each incident records:** every link's reason (machine/IP, gap, strong/weak), a confidence (high = all strong, medium = any weak, low = single group) and notes on what is uncertain.

**Numbers are starting values.** Tune on russellmitchell, santos, shaw; test once on fox, harrison, wardbeck, wheeler, wilson. Known risk (confirmed and fixed by tuning): 30 min was too short because cracking lasts ~37 min with few alerts.

*Review: _______________ (teammate) on ____________*

### Step 2.2a - Rule 1: merge bursts

In [9]:
# Settings agreed in Step 2.1 (starting values, to be tuned)
# 23 Nov: max_gap_s tuned 30 -> 45 min on the 3 tuning networks (see 'Tuning (a)')
CFG = dict(burst_gap_s=60, max_group_span_s=600,          # Rule 1
           baseline_hours=24, routine_min_bins=6, spike_factor=3,   # Rule 2
           max_gap_s=2700, strong_gap_s=300)               # Rule 3 (gap tuned to 45 min)


def merge_bursts(alerts, burst_gap_s=60, max_group_span_s=600):
    """Rule 1: repeats of the same alert type on the same machine, less than burst_gap_s apart,
    become one group; a group lasts at most max_group_span_s.
    Returns (groups, group_of_alert) - group_of_alert has the same index as alerts."""
    # 1) repeats of the same alert on the same machine next to each other, in time order
    a = alerts.sort_values(['host', 'short', 'ts', 'alert_id'], kind='stable').copy()

    # 2) a new group starts at a new machine/type, or after a gap longer than burst_gap_s
    same_type = (a['host'] == a['host'].shift()) & (a['short'] == a['short'].shift())
    gap = a['ts'].diff()
    new_run = ~same_type | (gap > burst_gap_s)

    # 3) number the runs
    run_id = new_run.cumsum()

    # 4) cap: cut every run into slices of at most max_group_span_s
    run_start = a.groupby(run_id)['ts'].transform('min')
    piece = (a['ts'] - run_start) // max_group_span_s
    a['group_key'] = run_id.astype(str) + '-' + piece.astype(str)

    # 5) one summary row per group
    groups = (a.groupby('group_key', sort=False)
                .agg(host=('host', 'first'), host_ip=('host_ip', 'first'),
                     short=('short', 'first'), signature=('signature', 'first'),
                     first_ts=('ts', 'min'), last_ts=('ts', 'max'),
                     count=('ts', 'size'), first_alert=('alert_id', 'first'))
                .sort_values('first_ts', kind='stable')
                .reset_index())
    groups['group_id'] = [f'grp-{i:06d}' for i in range(len(groups))]
    key_to_id = dict(zip(groups['group_key'], groups['group_id']))
    group_of_alert = a['group_key'].map(key_to_id).reindex(alerts.index)
    return groups.drop(columns='group_key'), group_of_alert

In [10]:
# Try Rule 1 on russellmitchell
sc = 'russellmitchell'
alerts = pd.read_parquet(OUT / f'{sc}_alerts.parquet')
groups, group_of_alert = merge_bursts(alerts, CFG['burst_gap_s'], CFG['max_group_span_s'])

# Checks: every alert is in exactly one group, nothing lost
assert group_of_alert.notna().all(), 'some alerts have no group'
assert groups['count'].sum() == len(alerts), 'group sizes do not add up'

print(f'{len(alerts):,} alerts -> {len(groups):,} groups '
      f'({len(alerts) / len(groups):.1f} alerts per group on average)')
print('Groups with a single alert:', int((groups["count"] == 1).sum()))
print('Longest group:', int((groups['last_ts'] - groups['first_ts']).max()), 'seconds (cap is', CFG['max_group_span_s'], ')')

biggest = groups.sort_values('count', ascending=False).head(8).copy()
biggest['start'] = pd.to_datetime(biggest['first_ts'], unit='s')
biggest['seconds'] = biggest['last_ts'] - biggest['first_ts']
biggest[['group_id', 'host', 'short', 'signature', 'count', 'start', 'seconds']]

45,544 alerts -> 4,437 groups (10.3 alerts per group on average)
Groups with a single alert: 636
Longest group: 591 seconds (cap is 600 )


,group_id,host,short,signature,count,start,seconds
3294,grp-003294,intranet_server,W-Acc-400,Wazuh: Web server 400 error code.,7068,2022-01-24 03:57:01,71
3333,grp-003333,intranet_server,A-Acc-Val1,AMiner: New request method in Apache Access log.,3169,2022-01-24 03:57:52,20
3331,grp-003331,intranet_server,W-All-Mul3,Wazuh: Multiple web server 400 error codes fro...,548,2022-01-24 03:57:26,46
2453,grp-002453,mail,W-Sys-Dov,Wazuh: Dovecot Authentication Success.,153,2022-01-23 09:19:09,591
1495,grp-001495,vpn,W-All-Evt,Wazuh: IDS event.,135,2022-01-22 10:24:14,12
3745,grp-003745,vpn,W-All-Evt,Wazuh: IDS event.,132,2022-01-24 10:38:41,7
3740,grp-003740,inet-firewall,W-All-Evt,Wazuh: IDS event.,126,2022-01-24 10:38:41,7
697,grp-000697,vpn,W-All-Evt,Wazuh: IDS event.,118,2022-01-21 12:58:30,309


### Step 2.2b - Rule 2: filter routine noise (learned from the first 24 h, no labels)

In [11]:
import numpy as np

SLOT_S = 600  # 10-minute slots


def routine_baseline(alerts, baseline_hours=24):
    """Learn 'normal' from the first baseline_hours: for every (machine, alert type),
    in how many 10-min slots it appeared and its busiest slot."""
    start = alerts['ts'].min()
    base = alerts[alerts['ts'] < start + baseline_hours * 3600]
    slot = (base['ts'] - start) // SLOT_S
    per_slot = base.assign(slot=slot).groupby(['host', 'short', 'slot']).size()
    return (per_slot.groupby(level=['host', 'short'])
                    .agg(baseline_slots='size', baseline_peak='max')
                    .reset_index())


def mark_routine(groups, baseline, routine_min_bins=6, spike_factor=3):
    """Rule 2: a group is routine if its type is regular on a normal day AND it is not a spike."""
    g = groups.merge(baseline, on=['host', 'short'], how='left')
    g[['baseline_slots', 'baseline_peak']] = g[['baseline_slots', 'baseline_peak']].fillna(0)
    regular = g['baseline_slots'] >= routine_min_bins
    spike = g['count'] > spike_factor * g['baseline_peak']
    g['routine'] = regular & ~spike
    g['why'] = np.select(
        [regular & ~spike, regular & spike, ~regular],
        ['routine: seen regularly on a normal day, at normal volume',
         'unusual: a normal alert type, but a spike over its normal volume',
         'unusual: not seen regularly on a normal day'],
        default='')
    return g

In [12]:
# Try Rule 2 on russellmitchell (uses the groups from Rule 1)
baseline = routine_baseline(alerts, CFG['baseline_hours'])
groups = mark_routine(groups, baseline, CFG['routine_min_bins'], CFG['spike_factor'])

print('Learned from day one:', len(baseline), '(machine, alert type) pairs,',
      int((baseline['baseline_slots'] >= CFG['routine_min_bins']).sum()), 'of them regular')
print()
print(groups['why'].value_counts().to_string())
print()
kept = groups[~groups['routine']]
print(f'Groups kept (unusual): {len(kept):,} of {len(groups):,}')
print(f'Alerts kept: {kept["count"].sum():,} of {len(alerts):,} ({kept["count"].sum() / len(alerts):.1%})')

# The 8 biggest groups again - which survive Rule 2?
big = groups.sort_values('count', ascending=False).head(8).copy()
big['start'] = pd.to_datetime(big['first_ts'], unit='s')
big[['host', 'short', 'count', 'start', 'baseline_slots', 'baseline_peak', 'routine', 'why']]

Learned from day one: 124 (machine, alert type) pairs, 24 of them regular

why
routine: seen regularly on a normal day, at normal volume           3969
unusual: not seen regularly on a normal day                          467
unusual: a normal alert type, but a spike over its normal volume       1

Groups kept (unusual): 468 of 4,437
Alerts kept: 12,118 of 45,544 (26.6%)


,host,short,count,start,baseline_slots,baseline_peak,routine,why
3294,intranet_server,W-Acc-400,7068,2022-01-24 03:57:01,0.0,0.0,False,unusual: not seen regularly on a normal day
3333,intranet_server,A-Acc-Val1,3169,2022-01-24 03:57:52,1.0,1.0,False,unusual: not seen regularly on a normal day
3331,intranet_server,W-All-Mul3,548,2022-01-24 03:57:26,0.0,0.0,False,unusual: not seen regularly on a normal day
2453,mail,W-Sys-Dov,153,2022-01-23 09:19:09,63.0,93.0,True,"routine: seen regularly on a normal day, at no..."
1495,vpn,W-All-Evt,135,2022-01-22 10:24:14,10.0,102.0,True,"routine: seen regularly on a normal day, at no..."
3745,vpn,W-All-Evt,132,2022-01-24 10:38:41,10.0,102.0,True,"routine: seen regularly on a normal day, at no..."
3740,inet-firewall,W-All-Evt,126,2022-01-24 10:38:41,40.0,254.0,True,"routine: seen regularly on a normal day, at no..."
697,vpn,W-All-Evt,118,2022-01-21 12:58:30,10.0,102.0,True,"routine: seen regularly on a normal day, at no..."


In [13]:
# ===== EVALUATION PEEK - reads the answers. NOT part of the agent. =====
labels = pd.read_parquet(OUT / f'{sc}_eval.parquet')
peek = alerts[['alert_id']].assign(group_id=group_of_alert.values).merge(labels, on='alert_id')
routine_ids = set(groups.loc[groups['routine'], 'group_id'])
peek['kept'] = ~peek['group_id'].isin(routine_ids)
peek['attack_by_time'] = peek['eval_phase'].isin(ATTACK_PHASES)   # inside an attack window (noisy)
peek['attack_exact'] = peek['eval_event_label'] != '-'            # matched to a real attack event

def kept_share(mask):
    return f"{peek.loc[mask, 'kept'].mean():.1%} of {int(mask.sum()):,}"

print('Attack alerts kept (by time window):', kept_share(peek['attack_by_time']))
print('Attack alerts kept (exact label):   ', kept_share(peek['attack_exact']))
print('False-alarm alerts kept:            ', kept_share(~peek['attack_by_time']))
print()
per_step = (peek[peek['attack_by_time']].groupby('eval_phase')['kept']
            .agg(alerts='size', kept='sum', share='mean'))
per_step.loc[[p for p in ATTACK_PHASES if p in per_step.index]].round(3)

Attack alerts kept (by time window): 93.7% of 12,015
Attack alerts kept (exact label):    100.0% of 10,962
False-alarm alerts kept:             2.6% of 33,529



,alerts,kept,share
eval_phase,,,
network_scans,8,0,0.000
service_scans,367,343,0.935
dirb,4522,4522,1.000
wpscan,6359,6357,1.000
webshell,5,2,0.400
cracking,14,4,0.286
reverse_shell,3,3,1.000
privilege_escalation,24,24,1.000
service_stop,2,2,1.000


### Step 2.2c - Rule 3: link only on shared evidence (+ Rule 4 placeholder)

In [14]:
def entities(group):
    """What a group can share with others: its machine and every IP we know."""
    ents = {f"host:{group['host']}"}
    if isinstance(group.get('host_ip'), str):
        ents.add(f"ip:{group['host_ip']}")
    for col in ('src_ips', 'dst_ips'):          # filled once we read the raw Zenodo file
        for ip in group.get(col) or []:
            ents.add(f'ip:{ip}')
    return ents


def order_ok(incident_groups, group):
    """Rule 4 placeholder: needs Omar's Kill Chain stages (O-3). Always yes for now."""
    return True


def link_groups(groups, max_gap_s=1800, strong_gap_s=300):
    """Rule 3: a group joins an incident only if it shares an entity with it AND starts
    <= max_gap_s after that entity was last seen. Returns (groups with incident_id, links)."""
    groups = groups.sort_values(['first_ts', 'group_id'], kind='stable').reset_index(drop=True)
    incident_of, links = [], []
    members = {}      # incident -> its groups
    last_seen = {}    # entity -> {'incident_id', 'last_ts', 'alert_id'}
    n = 0

    for row in groups.to_dict('records'):
        if row['routine']:                      # routine groups never link
            n += 1
            incident_of.append(f'inc-{n:06d}')
            continue

        ents = entities(row)
        best = None                             # (gap, entity, seen)
        for e in sorted(ents):
            seen = last_seen.get(e)
            if seen is None:
                continue
            gap = row['first_ts'] - seen['last_ts']
            if gap <= max_gap_s and order_ok(members[seen['incident_id']], row):
                if best is None or gap < best[0]:
                    best = (gap, e, seen)

        if best is None:                        # nothing shared within the gap -> new incident
            n += 1
            iid = f'inc-{n:06d}'
            members[iid] = [row]
        else:                                   # join the closest incident and record why
            gap, e, seen = best
            iid = seen['incident_id']
            members[iid].append(row)
            links.append({'incident_id': iid, 'alert_id': row['first_alert'],
                          'linked_to': seen['alert_id'], 'shared_entity': e,
                          'gap_seconds': int(max(gap, 0)),
                          'strength': 'strong' if gap <= strong_gap_s else 'weak'})
        incident_of.append(iid)

        for e in ents:                          # remember where each entity was last seen
            prev = last_seen.get(e)
            if prev is None or prev['incident_id'] != iid or row['last_ts'] >= prev['last_ts']:
                last_seen[e] = {'incident_id': iid, 'last_ts': row['last_ts'], 'alert_id': row['first_alert']}

    groups['incident_id'] = incident_of
    links = pd.DataFrame(links, columns=['incident_id', 'alert_id', 'linked_to',
                                         'shared_entity', 'gap_seconds', 'strength'])
    return groups, links


def summarise_incidents(groups, links):
    """One row per incident, with a confidence level."""
    weak = links[links['strength'] == 'weak'].groupby('incident_id').size()
    inc = groups.groupby('incident_id').agg(
        routine=('routine', 'all'), groups=('group_id', 'size'), alerts=('count', 'sum'),
        hosts=('host', lambda s: sorted(set(s))), first_ts=('first_ts', 'min'),
        last_ts=('last_ts', 'max'), types=('short', lambda s: sorted(set(s)))).reset_index()
    inc['weak_links'] = inc['incident_id'].map(weak).fillna(0).astype(int)
    inc['confidence'] = np.select([inc['routine'] | (inc['groups'] == 1), inc['weak_links'] > 0],
                                  ['low', 'medium'], default='high')
    return inc

In [15]:
# Try Rule 3 on russellmitchell (uses the groups marked by Rule 2)
groups, links = link_groups(groups, CFG['max_gap_s'], CFG['strong_gap_s'])
incidents = summarise_incidents(groups, links)
real = incidents[~incidents['routine']]

print(f"{int((~groups['routine']).sum())} unusual groups -> {len(real)} incidents")
print(f"Links made: {len(links)} ({(links['strength'] == 'strong').sum()} strong, {(links['strength'] == 'weak').sum()} weak)")
print('Confidence:', real['confidence'].value_counts().to_dict())
print('Incidents with more than one group:', int((real['groups'] > 1).sum()))

show = real.sort_values('alerts', ascending=False).head(10).copy()
show['start'] = pd.to_datetime(show['first_ts'], unit='s')
show['minutes'] = ((show['last_ts'] - show['first_ts']) / 60).round(1)
show[['incident_id', 'hosts', 'groups', 'alerts', 'start', 'minutes', 'weak_links', 'confidence']]

468 unusual groups -> 160 incidents
Links made: 308 (259 strong, 49 weak)
Confidence: {'low': 76, 'high': 55, 'medium': 29}
Incidents with more than one group: 84


,incident_id,hosts,groups,alerts,start,minutes,weak_links,confidence
3067,inc-003068,[intranet_server],24,10921,2022-01-24 03:56:47,41.3,1,medium
3071,inc-003072,[webserver],9,106,2022-01-24 03:56:53,0.4,0,high
3068,inc-003069,[mail],9,88,2022-01-24 03:56:47,0.5,0,high
103,inc-000104,[cloud_share],15,60,2022-01-21 06:08:23,26.9,1,medium
3274,inc-003275,[mail],17,51,2022-01-24 09:22:36,55.3,2,medium
3069,inc-003070,[morris_mail],3,44,2022-01-24 03:56:47,0.2,0,high
3076,inc-003077,[davey_mail],3,44,2022-01-24 03:57:02,0.3,0,high
62,inc-000063,[davey_mail],12,32,2022-01-21 05:20:12,26.7,1,medium
0,inc-000001,[cloud_share],6,28,2022-01-21 00:00:01,0.0,0,high
99,inc-000100,[intranet_server],13,28,2022-01-21 05:55:22,37.9,2,medium


In [16]:
# ===== EVALUATION PEEK 2 - reads the answers. NOT part of the agent. =====
# Which incidents did the real attack end up in?
peek['incident_id'] = peek['group_id'].map(groups.set_index('group_id')['incident_id'])
exact = peek[peek['kept'] & peek['attack_exact']]

per_inc = (exact.groupby('incident_id')
                .agg(attack_alerts=('alert_id', 'size'),
                     attack_steps=('eval_event_label', lambda s: sorted(set(s))))
                .join(incidents.set_index('incident_id')[['hosts', 'alerts', 'confidence']]))
per_inc['share_attack'] = (per_inc['attack_alerts'] / per_inc['alerts']).round(3)

print('Incidents holding exactly-labelled attack alerts:', len(per_inc))
print('Incidents holding any alert from an attack window:',
      peek.loc[peek['kept'] & peek['attack_by_time'], 'incident_id'].nunique())
print('Machines the attack touched:', sorted({h for hs in per_inc['hosts'] for h in hs}))
per_inc.sort_values('attack_alerts', ascending=False)

Incidents holding exactly-labelled attack alerts: 10
Incidents holding any alert from an attack window: 13
Machines the attack touched: ['cloud_share', 'inet-firewall', 'internal_share', 'intranet_server', 'mail', 'monitoring', 'vpn', 'webserver']


,attack_alerts,attack_steps,hosts,alerts,confidence,share_attack
incident_id,,,,,,
inc-003068,10913,"[attacker_change_user, dirb, escalated_sudo_co...",[intranet_server],10921,medium,0.999
inc-003074,15,[service_scan],[vpn],22,high,0.682
inc-003069,12,[service_scan],[mail],88,high,0.136
inc-003072,6,[service_scan],[webserver],106,high,0.057
inc-003067,5,[service_scan],[inet-firewall],8,high,0.625
inc-003078,3,[service_scan],[cloud_share],14,high,0.214
inc-003690,2,[dnsteal],[internal_share],2,low,1.000
inc-000120,1,[dnsteal],[inet-firewall],8,medium,0.125
inc-001083,1,[dnsteal],[inet-firewall],1,low,1.000


### Tuning (a) - the linking gap, on the 3 tuning networks only (russellmitchell, santos, shaw)

In [17]:
TUNE = ['russellmitchell', 'santos', 'shaw']
TEST = ['fox', 'harrison', 'wardbeck', 'wheeler', 'wilson']   # do NOT run these until the final test


def run_agent(sc, cfg):
    """The agent: Rules 1-3 on one network. Reads only the alerts table."""
    a = pd.read_parquet(OUT / f'{sc}_alerts.parquet')
    g, g_of_a = merge_bursts(a, cfg['burst_gap_s'], cfg['max_group_span_s'])
    g = mark_routine(g, routine_baseline(a, cfg['baseline_hours']), cfg['routine_min_bins'], cfg['spike_factor'])
    g, l = link_groups(g, cfg['max_gap_s'], cfg['strong_gap_s'])
    return a, g, g_of_a, l, summarise_incidents(g, l)


def score(sc, cfg):
    """EVALUATION ONLY: runs the agent, then compares with the answers."""
    a, g, g_of_a, l, inc = run_agent(sc, cfg)
    lab = pd.read_parquet(OUT / f'{sc}_eval.parquet')
    p = a[['alert_id']].assign(group_id=g_of_a.values).merge(lab, on='alert_id')
    p['incident_id'] = p['group_id'].map(g.set_index('group_id')['incident_id'])
    p['kept'] = ~p['incident_id'].isin(set(inc.loc[inc['routine'], 'incident_id']))
    p['attack'] = (p['eval_event_label'] != '-') & p['eval_phase'].isin(ATTACK_PHASES)  # exact AND in a window
    att = p[p['kept'] & p['attack']]
    counts = att['incident_id'].value_counts()
    top = counts.index[0]
    inside = p[p['incident_id'].isin(counts.index)]
    return {'network': sc, 'gap_min': cfg['max_gap_s'] // 60,
            'incidents': int((~inc['routine']).sum()),
            'attack_kept': round(len(att) / max(int(p['attack'].sum()), 1), 3),
            'attack_incidents': len(counts),
            'top_share': round(counts.iloc[0] / len(att), 3),
            'top_steps': ', '.join(sorted(set(att.loc[att['incident_id'] == top, 'eval_event_label']))),
            'purity': round(inside['attack'].mean(), 3),
            'weak_links': int((l['strength'] == 'weak').sum())}


rows = []
for gap_min in [30, 45, 60, 90]:
    cfg = {**CFG, 'max_gap_s': gap_min * 60}
    for sc in TUNE:
        rows.append(score(sc, cfg))
tuning = pd.DataFrame(rows)
tuning.to_csv(OUT / 'tuning_gap.csv', index=False)

pd.set_option('display.max_colwidth', 80)
tuning.sort_values(['network', 'gap_min'])

,network,gap_min,incidents,attack_kept,attack_incidents,top_share,top_steps,purity,weak_links
0,russellmitchell,30,169,1.0,9,0.994,"dirb, service_scan, webshell_cmd, wpscan",0.982,40
3,russellmitchell,45,160,1.0,8,0.996,"attacker_change_user, dirb, escalated_sudo_command, service_scan, webshell_c...",0.982,49
6,russellmitchell,60,149,1.0,8,0.996,"attacker_change_user, dirb, escalated_sudo_command, service_scan, webshell_c...",0.982,60
9,russellmitchell,90,122,1.0,8,0.996,"attacker_change_user, dirb, escalated_sudo_command, service_scan, webshell_c...",0.981,87
1,santos,30,203,1.0,8,0.994,"dirb, online_cracking, service_scan, webshell_cmd, wpscan",0.994,101
4,santos,45,183,1.0,7,0.998,"attacker_change_user, dirb, escalated_sudo_command, online_cracking, service...",0.994,121
7,santos,60,168,1.0,7,0.998,"attacker_change_user, dirb, escalated_sudo_command, online_cracking, service...",0.992,136
10,santos,90,146,1.0,7,0.998,"attacker_change_user, dirb, escalated_sudo_command, online_cracking, service...",0.991,158
2,shaw,30,249,1.0,5,0.991,"dirb, webshell_cmd, wpscan",1.000,93
5,shaw,45,229,1.0,4,0.998,"attacker_change_user, dirb, escalated_sudo_command, webshell_cmd, wpscan",1.000,113


In [18]:
# Same table, compact: one block per network
for sc, t in tuning.groupby('network'):
    print(f'=== {sc} ===')
    print(t.drop(columns=['network', 'top_steps']).to_string(index=False))
    for _, r in t.iterrows():
        print(f"  gap {r['gap_min']:>2} min -> biggest attack incident has: {r['top_steps']}")
    print()

=== russellmitchell ===
 gap_min  incidents  attack_kept  attack_incidents  top_share  purity  weak_links
      30        169          1.0                 9      0.994   0.982          40
      45        160          1.0                 8      0.996   0.982          49
      60        149          1.0                 8      0.996   0.982          60
      90        122          1.0                 8      0.996   0.981          87
  gap 30 min -> biggest attack incident has: dirb, service_scan, webshell_cmd, wpscan
  gap 45 min -> biggest attack incident has: attacker_change_user, dirb, escalated_sudo_command, service_scan, webshell_cmd, wpscan
  gap 60 min -> biggest attack incident has: attacker_change_user, dirb, escalated_sudo_command, service_scan, webshell_cmd, wpscan
  gap 90 min -> biggest attack incident has: attacker_change_user, dirb, escalated_sudo_command, service_scan, webshell_cmd, wpscan

=== santos ===
 gap_min  incidents  attack_kept  attack_incidents  top_share  purit

#### Decision (23 Nov): linking gap = **45 min** (was 30)

Tuned on russellmitchell, santos, shaw only. Everything else unchanged. "Attack" = exact label inside an attack window.

| Gap | Attack split into (rm / santos / shaw) | Privilege escalation in the main incident? | Purity (rm / santos / shaw) | Incidents (rm) | Weak links (rm) |
|---|---|---|---|---|---|
| 30 min | 9 / 8 / 5 | no | 0.982 / 0.994 / 1.000 | 169 | 40 |
| **45 min** | **8 / 7 / 4** | **yes, all 3** | 0.982 / 0.994 / 1.000 | 160 | 49 |
| 60 min | 8 / 7 / 4 | yes | 0.982 / 0.992 / 1.000 | 149 | 60 |
| 90 min | 8 / 7 / 4 | yes | 0.981 / 0.991 / 0.993 | 122 | 87 |

**Why 45:** it is the smallest gap that joins privilege escalation (changing user, sudo) to the main web-server incident in all three networks, with no loss of purity. Longer gaps add nothing for the attack - only merge unrelated normal incidents and add weak links, and at 90 min purity starts to fall. Attack alerts kept stays 100% in every run.

**Still split, and no gap can fix it:** the scan on other machines and the data theft hours later. Needs src/dst IPs (raw Zenodo file) or Omar's Kill Chain stages (decision b).

### Tuning (b) - the routine filter (routine_min_bins x spike_factor), tuning networks only, gap = 45 min

In [19]:
def score(sc, cfg):
    """EVALUATION ONLY: runs the agent, then compares with the answers. (adds false_alarms_kept)"""
    a, g, g_of_a, l, inc = run_agent(sc, cfg)
    lab = pd.read_parquet(OUT / f'{sc}_eval.parquet')
    p = a[['alert_id']].assign(group_id=g_of_a.values).merge(lab, on='alert_id')
    p['incident_id'] = p['group_id'].map(g.set_index('group_id')['incident_id'])
    p['kept'] = ~p['incident_id'].isin(set(inc.loc[inc['routine'], 'incident_id']))
    p['attack'] = (p['eval_event_label'] != '-') & p['eval_phase'].isin(ATTACK_PHASES)
    normal_time = ~p['eval_phase'].isin(ATTACK_PHASES)
    att = p[p['kept'] & p['attack']]
    counts = att['incident_id'].value_counts()
    inside = p[p['incident_id'].isin(counts.index)]
    return {'network': sc, 'gap_min': cfg['max_gap_s'] // 60,
            'min_bins': cfg['routine_min_bins'], 'spike': cfg['spike_factor'],
            'attack_kept': round(len(att) / max(int(p['attack'].sum()), 1), 4),
            'false_alarms_kept': round(p.loc[normal_time, 'kept'].mean(), 4),
            'alerts_kept': int(p['kept'].sum()),
            'incidents': int((~inc['routine']).sum()),
            'attack_incidents': len(counts),
            'purity': round(inside['attack'].mean(), 3)}


rows = []
for min_bins in [3, 6, 12]:
    for spike in [2, 3, 5]:
        cfg = {**CFG, 'routine_min_bins': min_bins, 'spike_factor': spike}
        for sc in TUNE:
            rows.append(score(sc, cfg))
tuning_b = pd.DataFrame(rows)
tuning_b.to_csv(OUT / 'tuning_routine.csv', index=False)

# One line per setting, averaged over the 3 tuning networks (worst case for attack_kept)
summary_b = (tuning_b.groupby(['min_bins', 'spike'])
             .agg(attack_kept_worst=('attack_kept', 'min'),
                  false_alarms_kept=('false_alarms_kept', 'mean'),
                  incidents=('incidents', 'mean'),
                  attack_incidents=('attack_incidents', 'mean'),
                  purity=('purity', 'mean'))
             .round(4).reset_index())
print(summary_b.to_string(index=False))
print()
print(tuning_b.sort_values(['network', 'min_bins', 'spike']).to_string(index=False))

 min_bins  spike  attack_kept_worst  false_alarms_kept  incidents  attack_incidents  purity
        3      2             0.9995             0.0291   131.3333            5.3333  0.9923
        3      3             0.9995             0.0202   130.3333            5.3333  0.9930
        3      5             0.9987             0.0154   128.0000            5.3333  0.9930
        6      2             0.9998             0.0353   191.3333            6.3333  0.9920
        6      3             0.9998             0.0266   190.6667            6.3333  0.9920
        6      5             0.9998             0.0221   188.6667            6.3333  0.9920
       12      2             1.0000             0.0811   234.6667            6.3333  0.9910
       12      3             1.0000             0.0735   234.6667            6.3333  0.9910
       12      5             1.0000             0.0697   234.6667            6.3333  0.9910

        network  gap_min  min_bins  spike  attack_kept  false_alarms_kept  aler

In [20]:
# EVALUATION ONLY: which real attack alerts does the filter throw away?
def lost_attack_alerts(sc, cfg):
    a, g, g_of_a, l, inc = run_agent(sc, cfg)
    lab = pd.read_parquet(OUT / f'{sc}_eval.parquet')
    p = a.assign(group_id=g_of_a.values).merge(lab, on='alert_id')
    p['incident_id'] = p['group_id'].map(g.set_index('group_id')['incident_id'])
    p['kept'] = ~p['incident_id'].isin(set(inc.loc[inc['routine'], 'incident_id']))
    p['attack'] = (p['eval_event_label'] != '-') & p['eval_phase'].isin(ATTACK_PHASES)
    lost = p[p['attack'] & ~p['kept']]
    return lost.groupby(['eval_event_label', 'host', 'short', 'signature']).size().rename('lost').reset_index()

for mb, sp in [(6, 3), (3, 3), (3, 5)]:
    print(f'=== routine_min_bins={mb}, spike_factor={sp} ===')
    for sc in TUNE:
        lost = lost_attack_alerts(sc, {**CFG, 'routine_min_bins': mb, 'spike_factor': sp})
        print(f'{sc}: {int(lost["lost"].sum())} lost')
        if len(lost):
            print(lost.to_string(index=False))
    print()

=== routine_min_bins=6, spike_factor=3 ===
russellmitchell: 1 lost
eval_event_label       host     short                                                   signature  lost
 crack_passwords monitoring A-Mon-Rng AMiner: CPU value out of expected range in monitoring logs.     1
santos: 1 lost
eval_event_label       host     short                                                   signature  lost
 crack_passwords monitoring A-Mon-Rng AMiner: CPU value out of expected range in monitoring logs.     1
shaw: 1 lost
eval_event_label       host     short                                                   signature  lost
 crack_passwords monitoring A-Mon-Rng AMiner: CPU value out of expected range in monitoring logs.     1

=== routine_min_bins=3, spike_factor=3 ===
russellmitchell: 5 lost
      eval_event_label            host      short                                                   signature  lost
  attacker_change_user intranet_server W-Aut-Pam1                           Wazuh: PAM: Login ses

#### Decision (23 Nov): routine filter stays at **routine_min_bins = 6, spike_factor = 3** (option A)

Tuned on russellmitchell, santos, shaw only, with the 45-min gap. "Attack" = exact label inside an attack window.

| min_bins | spike | Attack kept (worst network) | False alarms kept (avg) | Incidents (avg) |
|---|---|---|---|---|
| 3 | 2 / 3 / 5 | 99.95% / 99.95% / 99.87% | 2.9% / 2.0% / 1.5% | ~130 |
| **6** | 2 / **3** / 5 | 99.98% (all) | 3.5% / **2.7%** / 2.2% | ~190 |
| 12 | 2 / 3 / 5 | 100% | 8.1% / 7.4% / 7.0% | ~235 |

**What each setting throws away (checked alert by alert):**
- **6 / 3:** 1 alert per network - *AMiner: CPU value out of expected range* on the monitoring server during password cracking. CPU alerts also happen on normal days, and one alert cannot be a spike.
- **3:** loses **privilege-escalation evidence** (*PAM: Login session opened/closed* on the web server) - rejected.
- **12:** loses nothing, but the analyst sees ~3x more false alarms - rejected (too costly for one weak CPU alert).

**Why A (6 / 3) over B (6 / 5):** same single lost alert; B removes slightly more false alarms (2.2% vs 2.7%) but a higher spike threshold could miss smaller attack bursts in the unseen test networks. A keeps what already works.

**Known limitation for the report:** password cracking leaves almost no unusual alerts (it runs on the attacker's machine); the one CPU alert it causes looks like normal monitoring noise.

**Final settings for the test run:** burst_gap 60 s, group cap 10 min, baseline 24 h, routine_min_bins 6, spike_factor 3, linking gap 45 min, strong link <= 5 min.

## Stage 3 - Step 3.1: unit tests

Tiny hand-made examples where we already know the right answer - one test per sentence of the 2.1 rules. They use the real functions above with fake data; the "normal day" is shortened to **1 hour** (and "regular" to 3 slots) so the examples stay small. Every other setting is the tuned one (45-min gap, 5-min strong link). Run the runner cell below after any change to the rules: every line must say PASS.

In [21]:
# ---------- helpers: build tiny fake networks ----------
T0 = 1_700_000_000          # any start time
H = 2 * 3600                # 'attack' alerts happen 2 h in, after the 1-hour normal day
IPS = {'web': '10.0.0.1', 'mail': '10.0.0.2', 'fw': '10.0.0.3'}
TEST_CFG = {**CFG, 'baseline_hours': 1, 'routine_min_bins': 3}

# The normal day: the mail server's antivirus update every 10 minutes (6 slots -> regular)
NORMAL_DAY = [(i * 600, 'mail', 'W-Sys-Cav') for i in range(6)]


def make_alerts(rows):
    """rows = (seconds after T0, machine, alert type)"""
    a = pd.DataFrame(rows, columns=['dt', 'host', 'short'])
    a['ts'] = T0 + a['dt']
    a = a.sort_values('ts', kind='stable').reset_index(drop=True)
    a['alert_id'] = [f'alert-t-{i:05d}' for i in range(len(a))]
    a['host_ip'] = a['host'].map(IPS)
    a['signature'] = 'test: ' + a['short']
    a['network'] = 't'
    return a.drop(columns='dt')


def pipeline(a, shared_ips=None, **change):
    """Rules 1-3 exactly as the agent runs them. shared_ips = {machine: [extra IPs]} for the IP test."""
    cfg = {**TEST_CFG, **change}
    g, g_of_a = merge_bursts(a, cfg['burst_gap_s'], cfg['max_group_span_s'])
    if shared_ips:
        g['src_ips'] = g['host'].map(lambda h: shared_ips.get(h, []))
    g = mark_routine(g, routine_baseline(a, cfg['baseline_hours']), cfg['routine_min_bins'], cfg['spike_factor'])
    g, links = link_groups(g, cfg['max_gap_s'], cfg['strong_gap_s'])
    inc = summarise_incidents(g, links)
    a = a.assign(incident_id=g_of_a.map(g.set_index('group_id')['incident_id']).values)
    return a, g, links, inc


def incident_of(a, short):
    return a.loc[a['short'] == short, 'incident_id'].iloc[0]


# ---------- Rule 0: the answers stay out of the agent's table ----------
def test_loader_keeps_answers_out():
    path = '/tmp/x_alerts.txt'
    with open(path, 'w') as f:
        f.write('time,name,ip,host,short,time_label,event_label' + chr(10))
        f.write('1642723350,Suricata: Alert - ET POLICY test,10.0.0.1,web,S-Tst,wpscan,wpscan' + chr(10))
        f.write('1642723347,Wazuh: ClamAV database update,10.0.0.2,mail,W-Sys-Cav,false_positive,-' + chr(10))
        f.write('1642723360,AMiner: New event type.,10.0.0.3,fw,A-Tst,dirb,-' + chr(10))
    alerts, labels = load_alert_csv(path, 'x')
    leaked = [c for c in alerts.columns if 'label' in c or c.startswith('eval_')]
    assert leaked == [], f'answer columns in the agent table: {leaked}'
    assert set(labels.columns) == {'alert_id', 'eval_phase', 'eval_event_label'}
    assert list(alerts['ts']) == sorted(alerts['ts']), 'alerts are not in time order'
    assert list(alerts['ids_source']) == ['wazuh', 'suricata', 'aminer']
    assert alerts['host_ip'].notna().all(), 'an IP was dropped'


# ---------- Rule 1: merge bursts ----------
def test_repeats_within_60s_merge():
    a = make_alerts([(H, 'web', 'A'), (H + 30, 'web', 'A'), (H + 59, 'web', 'A')])
    g, _ = merge_bursts(a)
    assert len(g) == 1 and g['count'].iloc[0] == 3, f'{len(g)} groups'


def test_gap_over_60s_splits():
    a = make_alerts([(H, 'web', 'A'), (H + 120, 'web', 'A')])
    g, _ = merge_bursts(a)
    assert len(g) == 2, f'{len(g)} groups'


def test_endless_stream_is_capped_at_10_min():
    a = make_alerts([(H + i * 30, 'web', 'A') for i in range(60)])   # every 30 s for 30 min
    g, _ = merge_bursts(a)
    assert len(g) == 3, f'{len(g)} groups'
    assert (g['last_ts'] - g['first_ts']).max() < 600


def test_different_types_not_merged():
    a = make_alerts([(H, 'web', 'A'), (H, 'web', 'B')])
    g, _ = merge_bursts(a)
    assert len(g) == 2, f'{len(g)} groups'


# ---------- Rule 2: routine filter ----------
def test_regular_type_at_normal_volume_is_routine():
    a, g, _, _ = pipeline(make_alerts(NORMAL_DAY + [(H, 'mail', 'W-Sys-Cav')]))
    assert g.loc[g['first_ts'] == T0 + H, 'routine'].iloc[0]


def test_spike_of_regular_type_is_kept():
    a, g, _, _ = pipeline(make_alerts(NORMAL_DAY + [(H + i, 'mail', 'W-Sys-Cav') for i in range(10)]))
    row = g.loc[g['first_ts'] == T0 + H].iloc[0]
    assert not row['routine'] and 'spike' in row['why'], row['why']


def test_type_never_seen_is_kept():
    a, g, _, _ = pipeline(make_alerts(NORMAL_DAY + [(H, 'web', 'X')]))
    row = g.loc[g['short'] == 'X'].iloc[0]
    assert not row['routine'] and 'not seen' in row['why'], row['why']


# ---------- Rule 3: link only on shared evidence ----------
def test_same_machine_2_min_apart_linked_strong():
    a, g, links, _ = pipeline(make_alerts(NORMAL_DAY + [(H, 'web', 'A'), (H + 120, 'web', 'B')]))
    assert incident_of(a, 'A') == incident_of(a, 'B')
    assert list(links['strength']) == ['strong'], list(links['strength'])


def test_different_machines_5s_apart_stay_separate():
    a, g, links, _ = pipeline(make_alerts(NORMAL_DAY + [(H, 'web', 'A'), (H + 5, 'fw', 'B')]))
    assert incident_of(a, 'A') != incident_of(a, 'B'), 'linked on time alone'
    assert links.empty


def test_gap_of_exactly_45_min_links():
    a, g, links, _ = pipeline(make_alerts(NORMAL_DAY + [(H, 'web', 'A'), (H + 45 * 60, 'web', 'B')]))
    assert incident_of(a, 'A') == incident_of(a, 'B')


def test_gap_over_45_min_stays_separate():
    a, g, links, _ = pipeline(make_alerts(NORMAL_DAY + [(H, 'web', 'A'), (H + 45 * 60 + 1, 'web', 'B')]))
    assert incident_of(a, 'A') != incident_of(a, 'B')


def test_10_min_gap_is_a_weak_link():
    a, g, links, _ = pipeline(make_alerts(NORMAL_DAY + [(H, 'web', 'A'), (H + 600, 'web', 'B')]))
    assert incident_of(a, 'A') == incident_of(a, 'B')
    assert list(links['strength']) == ['weak'], list(links['strength'])


def test_routine_alert_never_links():
    a, g, links, _ = pipeline(make_alerts(NORMAL_DAY + [(H, 'mail', 'A'), (H + 60, 'mail', 'W-Sys-Cav')]))
    routine_inc = a.loc[(a['short'] == 'W-Sys-Cav') & (a['ts'] == T0 + H + 60), 'incident_id'].iloc[0]
    assert routine_inc != incident_of(a, 'A')
    assert links.empty


def test_shared_attacker_ip_links_two_machines():
    attacker = {'web': ['203.0.113.66'], 'fw': ['203.0.113.66']}
    a, g, links, _ = pipeline(make_alerts(NORMAL_DAY + [(H, 'web', 'A'), (H + 60, 'fw', 'B')]), shared_ips=attacker)
    assert incident_of(a, 'A') == incident_of(a, 'B')
    assert list(links['shared_entity']) == ['ip:203.0.113.66'], list(links['shared_entity'])


# ---------- confidence ----------
def test_confidence_levels():
    def conf(rows):
        a, g, links, inc = pipeline(make_alerts(NORMAL_DAY + rows))
        return inc.set_index('incident_id').loc[incident_of(a, 'A'), 'confidence']
    assert conf([(H, 'web', 'A')]) == 'low'                            # single group
    assert conf([(H, 'web', 'A'), (H + 120, 'web', 'B')]) == 'high'     # all links strong
    assert conf([(H, 'web', 'A'), (H + 600, 'web', 'B')]) == 'medium'   # a weak link

print('16 tests defined')

16 tests defined


In [22]:
# ---------- runner: PASS / FAIL for every test ----------
tests = [(name, f) for name, f in list(globals().items()) if name.startswith('test_') and callable(f)]
passed = 0
for name, f in tests:
    try:
        f()
        print('PASS ', name)
        passed += 1
    except AssertionError as e:
        print('FAIL ', name, '->', e)
    except Exception as e:
        print('ERROR', name, '->', type(e).__name__, e)
print()
print(f'{passed} of {len(tests)} tests passed')

PASS  test_loader_keeps_answers_out
PASS  test_repeats_within_60s_merge
PASS  test_gap_over_60s_splits
PASS  test_endless_stream_is_capped_at_10_min
PASS  test_different_types_not_merged
PASS  test_regular_type_at_normal_volume_is_routine
PASS  test_spike_of_regular_type_is_kept
PASS  test_type_never_seen_is_kept
PASS  test_same_machine_2_min_apart_linked_strong
PASS  test_different_machines_5s_apart_stay_separate
PASS  test_gap_of_exactly_45_min_links
PASS  test_gap_over_45_min_stays_separate
PASS  test_10_min_gap_is_a_weak_link
PASS  test_routine_alert_never_links
PASS  test_shared_attacker_ip_links_two_machines
PASS  test_confidence_levels

16 of 16 tests passed


## Stage 3 - Step 3.2: demo scenario 1 (web attack) with ground truth

A small invented network. **Day 1** is a normal day (the agent learns "normal" from it). **Day 2, 10:00:** outside attacker 203.0.113.66 attacks.

| # | Step | Machine | Alerts | Technique | Kill Chain stage |
|---|---|---|---|---|---|
| 1 | network scan | inet-firewall | 20 | T1595 | reconnaissance |
| - | *coincidence: employee mistypes a mail password 5 s later* | mail | 1 | benign | - |
| 2 | service scan | webserver | 50 | T1595 | reconnaissance |
| 3 | wordlist scan (dirb / wpscan) | webserver | 200 | T1595.003 | reconnaissance |
| 4 | upload through WordPress | webserver | 1 | T1190 | exploitation |
| 5 | web shell starts | webserver | 1 | T1505.003 | installation |
| 6 | commands through the shell | webserver | 5 | T1059 | command & control |
| 7 | sudo to root (~29 min later) | webserver | 3 | T1548.003 | Omar to confirm |
| 8 | data theft over DNS (6 h later) | inet-dns | 40 | T1048.003 | actions on objectives |

**Ground truth:** GT-1 = steps 1-7 (High). GT-2 = step 8, same story, ideally a weak link to GT-1 - decision (b) (High). benign-1 = the typo, must stay alone (Low).

**Predicted for agent v1 (written before running):** 4 non-routine incidents - firewall scan alone; ONE web-server incident with steps 2-7, privilege escalation joined by a WEAK link; DNS theft separate; typo alone; background all routine.

**Files** (`scenarios/`): `scenario1_alerts.csv` (what the agent reads), `scenario1_truth.csv` (answers per alert), `scenario1_expected.json` (story, techniques, severity, v1 predictions). For Habiba (end-to-end tests), Omar (mapping check) and Lama (report).

In [23]:
import json

SCN = Path('scenarios')
SCN.mkdir(exist_ok=True)
ATTACKER = '203.0.113.66'
MACHINE_IP = {'webserver': '10.0.1.20', 'inet-firewall': '10.0.1.1', 'mail': '10.0.1.25',
              'inet-dns': '10.0.1.53', 'vpn': '10.0.1.10'}


def build_scenario_1():
    """Scenario 1 - web attack. Day 1 normal, day 2 the attack. Returns (alerts, truth)."""
    rows = []

    def add(dt, host, tool, short, sig, src_ip=None, dst_ip=None,
            gt='normal', step='normal', technique=None, stage=None):
        rows.append(dict(dt=dt, host=host, ids_source=tool, short=short, signature=sig,
                         src_ip=src_ip, dst_ip=dst_ip, gt_incident=gt, gt_step=step,
                         gt_technique=technique, gt_stage=stage))

    # --- normal background on both days ---
    for t in range(0, 2 * 86400, 600):
        add(t, 'mail', 'wazuh', 'W-Sys-Cav', 'Wazuh: ClamAV database update')
    for t in range(300, 2 * 86400, 1800):
        add(t, 'vpn', 'wazuh', 'W-All-Evt', 'Wazuh: IDS event.')
        add(t + 5, 'inet-firewall', 'wazuh', 'W-All-Evt', 'Wazuh: IDS event.')

    A = 86400 + 10 * 3600            # day 2, 10:00 UTC
    web = MACHINE_IP['webserver']
    # 1. network scan, seen by the firewall
    for i in range(20):
        add(A + 2 * i, 'inet-firewall', 'suricata', 'S-Scn-Nmp', 'Suricata: Alert - ET SCAN Possible Nmap scan',
            ATTACKER, f'10.0.1.{1 + i}', 'GT-1', 'network_scan', 'T1595', 'reconnaissance')
    # coincidence: an employee mistypes a mail password 5 s later (benign)
    add(A + 5, 'mail', 'wazuh', 'W-Mai-Inv', 'Wazuh: Dovecot invalid user login attempt.',
        '10.0.2.15', MACHINE_IP['mail'], 'benign-1', 'typo_login')
    # 2. service scan of the web server
    for i in range(50):
        add(A + 120 + i, 'webserver', 'wazuh', 'W-Acc-400', 'Wazuh: Web server 400 error code.',
            ATTACKER, web, 'GT-1', 'service_scan', 'T1595', 'reconnaissance')
    # 3. wordlist scanning (dirb / wpscan)
    for i in range(100):
        add(A + 240 + i, 'webserver', 'aminer', 'A-Acc-Val1', 'AMiner: New request method in Apache access log.',
            ATTACKER, web, 'GT-1', 'wordlist_scan', 'T1595.003', 'reconnaissance')
        add(A + 240 + i, 'webserver', 'wazuh', 'W-Acc-404', 'Wazuh: Web server 404 error code.',
            ATTACKER, web, 'GT-1', 'wordlist_scan', 'T1595.003', 'reconnaissance')
    # 4. exploit: file uploaded through the WordPress plugin; 5. the web shell starts
    add(A + 400, 'webserver', 'wazuh', 'W-Web-Upl', 'Wazuh: File added to the web directory.',
        ATTACKER, web, 'GT-1', 'upload_exploit', 'T1190', 'exploitation')
    add(A + 402, 'webserver', 'aminer', 'A-Aud-New', 'AMiner: New process in audit logs.',
        None, None, 'GT-1', 'web_shell', 'T1505.003', 'installation')
    # 6. commands through the web shell
    for i in range(5):
        add(A + 480 + 40 * i, 'webserver', 'aminer', 'A-Aud-Cmd', 'AMiner: New command in audit logs.',
            None, None, 'GT-1', 'commands', 'T1059', 'command_and_control')
    # 7. privilege escalation ~29 min after the last command
    add(A + 2400, 'webserver', 'wazuh', 'W-Aut-Pam1', 'Wazuh: PAM: Login session opened.',
        None, None, 'GT-1', 'privilege_escalation', 'T1548.003', 'to_confirm')
    for i in range(2):
        add(A + 2401 + 10 * i, 'webserver', 'wazuh', 'W-Aut-Sudo', 'Wazuh: Successful sudo to ROOT executed.',
            None, None, 'GT-1', 'privilege_escalation', 'T1548.003', 'to_confirm')
    # 8. data theft over DNS, 6 h later, seen by the DNS server (web server -> DNS server)
    for i in range(40):
        add(A + 6 * 3600 + 30 * i, 'inet-dns', 'suricata', 'S-Dns-Lng',
            'Suricata: Alert - ET DNS Query with unusually long subdomain',
            web, MACHINE_IP['inet-dns'], 'GT-2', 'dns_exfiltration', 'T1048.003', 'actions_on_objectives')

    t0 = int(pd.Timestamp('2026-01-10', tz='UTC').timestamp())
    df = pd.DataFrame(rows)
    df['ts'] = t0 + df['dt']
    df = df.sort_values('ts', kind='stable').reset_index(drop=True)
    df['alert_id'] = [f'alert-scenario1-{i:05d}' for i in range(len(df))]
    df['network'] = 'scenario1'
    df['time'] = pd.to_datetime(df['ts'], unit='s', utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    df['host_ip'] = df['host'].map(MACHINE_IP)
    agent_cols = ['alert_id', 'network', 'ts', 'time', 'ids_source', 'signature', 'short',
                  'host', 'host_ip', 'src_ip', 'dst_ip']
    truth_cols = ['alert_id', 'gt_incident', 'gt_step', 'gt_technique', 'gt_stage']
    return df[agent_cols], df[truth_cols]


expected_1 = {
    'scenario': 'scenario1_web_attack',
    'story': ('Outside attacker 203.0.113.66 scans the network, attacks the WordPress web server, uploads a '
              'web shell, runs commands, escalates to root, and 6 h later leaks data over DNS. An employee '
              'mistypes a mail password 5 s after the scan starts.'),
    'ground_truth': {
        'GT-1': {'what': 'web attack chain',
                 'steps': ['network_scan', 'service_scan', 'wordlist_scan', 'upload_exploit', 'web_shell',
                           'commands', 'privilege_escalation'],
                 'techniques': ['T1595', 'T1595.003', 'T1190', 'T1505.003', 'T1059', 'T1548.003'],
                 'kill_chain': ['reconnaissance', 'exploitation', 'installation', 'command_and_control'],
                 'expected_severity': 'High',
                 'why': 'multi-stage attack on the public web server that reaches root'},
        'GT-2': {'what': 'data theft over DNS from the web server', 'steps': ['dns_exfiltration'],
                 'techniques': ['T1048.003'], 'kill_chain': ['actions_on_objectives'],
                 'expected_severity': 'High',
                 'note': 'same attack story as GT-1; ideally linked to it by a weak link (decision b)'},
        'benign-1': {'what': 'employee mistypes a mail password', 'techniques': [], 'expected_severity': 'Low',
                     'note': 'must never be joined to the attack: other machine, only close in time'}},
    'expected_agent_v1': {
        'non_routine_incidents': 4,
        'GT-1': ('2 incidents: the firewall scan alone, and one web-server incident with all other steps; '
                 'privilege escalation joined by a WEAK link (~29 min gap)'),
        'GT-2': '1 incident on inet-dns, separate from GT-1 (other machine, 6 h later)',
        'benign-1': '1 incident of its own',
        'background': 'all routine, no incident'},
    'notes': ['Kill Chain stages are first guesses; Omar to confirm (privilege escalation has no stage of its own).',
              'Settings: the tuned v1 settings (CFG).'],
}

scn1_alerts, scn1_truth = build_scenario_1()
scn1_alerts.to_csv(SCN / 'scenario1_alerts.csv', index=False)
scn1_truth.to_csv(SCN / 'scenario1_truth.csv', index=False)
with open(SCN / 'scenario1_expected.json', 'w') as f:
    json.dump(expected_1, f, indent=2)

print(len(scn1_alerts), 'alerts written to', SCN.resolve())
print(scn1_truth['gt_incident'].value_counts().to_string())
scn1_alerts.merge(scn1_truth, on='alert_id').query("gt_incident != 'normal'").groupby(
    ['gt_incident', 'gt_step', 'host'], sort=False).size().rename('alerts').reset_index()

801 alerts written to /content/drive/MyDrive/sentinelai/scenarios
gt_incident
normal      480
GT-1        280
GT-2         40
benign-1      1


,gt_incident,gt_step,host,alerts
0,GT-1,network_scan,inet-firewall,20
1,benign-1,typo_login,mail,1
2,GT-1,service_scan,webserver,50
3,GT-1,wordlist_scan,webserver,200
4,GT-1,upload_exploit,webserver,1
5,GT-1,web_shell,webserver,1
6,GT-1,commands,webserver,5
7,GT-1,privilege_escalation,webserver,3
8,GT-2,dns_exfiltration,inet-dns,40


In [24]:
def agent(a, cfg=CFG):
    """The Investigation Agent v1 on any alerts table (Rules 1-3, tuned settings). Reads no answers."""
    g, g_of_a = merge_bursts(a, cfg['burst_gap_s'], cfg['max_group_span_s'])
    g = mark_routine(g, routine_baseline(a, cfg['baseline_hours']), cfg['routine_min_bins'], cfg['spike_factor'])
    g, links = link_groups(g, cfg['max_gap_s'], cfg['strong_gap_s'])
    inc = summarise_incidents(g, links)
    a = a.assign(incident_id=g_of_a.map(g.set_index('group_id')['incident_id']).values)
    return a, g, links, inc


# 1) the agent runs on the alerts only
a1, g1, l1, inc1 = agent(scn1_alerts)

# 2) EVALUATION: compare with the answers
t = a1.merge(scn1_truth, on='alert_id')
t['kept'] = ~t['incident_id'].isin(set(inc1.loc[inc1['routine'], 'incident_id']))

print('What the agent made of each true step:')
view = (t[t['gt_incident'] != 'normal']
        .groupby(['gt_incident', 'gt_step'], sort=False)
        .agg(alerts=('alert_id', 'size'), kept=('kept', 'sum'),
             agent_incident=('incident_id', lambda s: ', '.join(sorted(set(s))))))
print(view.to_string())
print()


def incs(gt, step=None):
    m = t['gt_incident'] == gt
    if step:
        m &= t['gt_step'] == step
    return set(t.loc[m, 'incident_id'])


web_steps = ['service_scan', 'wordlist_scan', 'upload_exploit', 'web_shell', 'commands', 'privilege_escalation']
web_inc = set().union(*[incs('GT-1', s) for s in web_steps])
priv_first = t.loc[t['gt_step'] == 'privilege_escalation', 'alert_id'].iloc[0]
checks = {
    'background is all routine': not t.loc[t['gt_incident'] == 'normal', 'kept'].any(),
    'every attack and benign alert kept': bool(t.loc[t['gt_incident'] != 'normal', 'kept'].all()),
    '4 non-routine incidents': int((~inc1['routine']).sum()) == 4,
    'web-server steps 2-7 form ONE incident': len(web_inc) == 1,
    'firewall scan is a separate incident': len(incs('GT-1', 'network_scan')) == 1 and incs('GT-1', 'network_scan').isdisjoint(web_inc),
    'privilege escalation joined by a WEAK link': bool((l1.loc[l1['alert_id'] == priv_first, 'strength'] == 'weak').any()),
    'DNS theft is one separate incident': len(incs('GT-2')) == 1 and incs('GT-2').isdisjoint(web_inc),
    'typo stays alone (coincidence not linked)': len(incs('benign-1')) == 1 and bool(t.loc[t['incident_id'].isin(incs('benign-1')), 'gt_incident'].eq('benign-1').all()),
}
for name, ok in checks.items():
    print('OK  ' if ok else 'DIFF', name)
print()
print(sum(checks.values()), 'of', len(checks), 'predictions met')

show = inc1[~inc1['routine']].copy()
show['start'] = pd.to_datetime(show['first_ts'], unit='s')
show[['incident_id', 'hosts', 'groups', 'alerts', 'start', 'weak_links', 'confidence', 'types']]

What the agent made of each true step:
                                  alerts  kept agent_incident
gt_incident gt_step                                          
GT-1        network_scan              20    20     inc-000341
benign-1    typo_login                 1     1     inc-000343
GT-1        service_scan              50    50     inc-000344
            wordlist_scan            200   200     inc-000344
            upload_exploit             1     1     inc-000344
            web_shell                  1     1     inc-000344
            commands                   5     5     inc-000344
            privilege_escalation       3     3     inc-000344
GT-2        dns_exfiltration          40    40     inc-000404

OK   background is all routine
OK   every attack and benign alert kept
OK   4 non-routine incidents
OK   web-server steps 2-7 form ONE incident
OK   firewall scan is a separate incident
OK   privilege escalation joined by a WEAK link
OK   DNS theft is one separate incident
OK  

,incident_id,hosts,groups,alerts,start,weak_links,confidence,types
340,inc-000341,[inet-firewall],1,20,2026-01-11 10:00:00,0,low,[S-Scn-Nmp]
342,inc-000343,[mail],1,1,2026-01-11 10:00:05,0,low,[W-Mai-Inv]
343,inc-000344,[webserver],8,260,2026-01-11 10:02:00,1,medium,"[A-Acc-Val1, A-Aud-Cmd, A-Aud-New, W-Acc-400, W-Acc-404, W-Aut-Pam1, W-Aut-S..."
403,inc-000404,[inet-dns],2,40,2026-01-11 16:00:00,0,high,[S-Dns-Lng]


## Stage 3 - Step 3.3: demo scenario 2 (innocent but noisy day) with ground truth

No attacker. Normal things that look suspicious. **Goal for the whole system: nothing rated High.**

| Event | When | Machine | Looks like | Truth |
|---|---|---|---|---|
| monitoring scan | every hour, both days | inet-firewall | the **same Nmap-scan alert** as the attack in scenario 1 | routine IT check |
| nightly backup | 02:00 both nights, 15 min | internal_share | burst of large-file-transfer alerts | normal backup |
| software update + sudo | day 2, 11:00 | webserver | packages installed, then **sudo to root** | admin maintenance |
| antivirus full refresh | day 2, 11:03 | mail | 60 ClamAV alerts in 1 min (normally 1 per 10 min) | antivirus |

**Predicted for agent v1 (written before running):** monitoring scan = routine (it is regular on day one); backup = 1 incident per night (2) - a once-a-day job shows in only 2 of the 6 needed slots, known limitation; update + sudo = 1 incident, strong link (alerts cannot show intent); antivirus refresh = its own incident (spike), NOT linked to the update 3 min earlier (other machine). **4 incidents, all expected severity Low.**

**Files:** `scenarios/scenario2_alerts.csv`, `scenario2_truth.csv`, `scenario2_expected.json`.

In [25]:
MACHINE_IP.update({'monitoring': '10.0.1.5', 'internal_share': '10.0.1.40', 'backup': '10.0.1.30'})


def build_scenario_2():
    """Scenario 2 - innocent but noisy. No attacker. Returns (alerts, truth)."""
    rows = []

    def add(dt, host, tool, short, sig, src_ip=None, dst_ip=None, gt='normal'):
        rows.append(dict(dt=dt, host=host, ids_source=tool, short=short, signature=sig,
                         src_ip=src_ip, dst_ip=dst_ip, gt_incident=gt, gt_step=gt,
                         gt_technique=None, gt_stage=None))

    # --- normal background on both days (same as scenario 1) ---
    for t in range(0, 2 * 86400, 600):
        add(t, 'mail', 'wazuh', 'W-Sys-Cav', 'Wazuh: ClamAV database update')
    for t in range(300, 2 * 86400, 1800):
        add(t, 'vpn', 'wazuh', 'W-All-Evt', 'Wazuh: IDS event.')
        add(t + 5, 'inet-firewall', 'wazuh', 'W-All-Evt', 'Wazuh: IDS event.')

    # monitoring scan every hour, both days - the SAME alert type as the attack scan in scenario 1
    for h in range(48):
        for i in range(10):
            add(h * 3600 + 900 + 2 * i, 'inet-firewall', 'suricata', 'S-Scn-Nmp',
                'Suricata: Alert - ET SCAN Possible Nmap scan',
                MACHINE_IP['monitoring'], f'10.0.1.{20 + i}', 'benign-monitoring')
    # nightly backup at 02:00, 15 minutes, both nights
    for night in (1, 2):
        for i in range(30):
            add((night - 1) * 86400 + 2 * 3600 + 30 * i, 'internal_share', 'wazuh', 'W-Sys-Big',
                'Wazuh: Large file transfer to file share.',
                MACHINE_IP['backup'], MACHINE_IP['internal_share'], f'benign-backup-{night}')
    D2 = 86400
    # software update on the web server, then the admin uses sudo
    for i in range(8):
        add(D2 + 11 * 3600 + 15 * i, 'webserver', 'wazuh', 'W-Pkg-Ins',
            'Wazuh: New dpkg (Debian Package) installed.', gt='benign-update')
    for i in range(2):
        add(D2 + 11 * 3600 + 150 + 10 * i, 'webserver', 'wazuh', 'W-Aut-Sudo',
            'Wazuh: Successful sudo to ROOT executed.', gt='benign-update')
    # antivirus full refresh on the mail server, 3 minutes later (a spike of a routine type)
    for i in range(60):
        add(D2 + 11 * 3600 + 180 + i, 'mail', 'wazuh', 'W-Sys-Cav',
            'Wazuh: ClamAV database update', gt='benign-av-refresh')

    t0 = int(pd.Timestamp('2026-01-20', tz='UTC').timestamp())
    df = pd.DataFrame(rows)
    df['ts'] = t0 + df['dt']
    df = df.sort_values('ts', kind='stable').reset_index(drop=True)
    df['alert_id'] = [f'alert-scenario2-{i:05d}' for i in range(len(df))]
    df['network'] = 'scenario2'
    df['time'] = pd.to_datetime(df['ts'], unit='s', utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    df['host_ip'] = df['host'].map(MACHINE_IP)
    agent_cols = ['alert_id', 'network', 'ts', 'time', 'ids_source', 'signature', 'short',
                  'host', 'host_ip', 'src_ip', 'dst_ip']
    truth_cols = ['alert_id', 'gt_incident', 'gt_step', 'gt_technique', 'gt_stage']
    return df[agent_cols], df[truth_cols]


expected_2 = {
    'scenario': 'scenario2_innocent_noisy_day',
    'story': ('No attacker. The monitoring server scans the network every hour, a backup runs every night at '
              '02:00, an admin installs updates and uses sudo on the web server, and the antivirus on the mail '
              'server does a full refresh 3 minutes later.'),
    'ground_truth': {
        'benign-monitoring': {'what': 'hourly monitoring scan (same alert type as the scenario-1 attack scan)',
                              'expected_severity': 'none (routine)'},
        'benign-backup-1': {'what': 'nightly backup, night 1', 'expected_severity': 'Low'},
        'benign-backup-2': {'what': 'nightly backup, night 2', 'expected_severity': 'Low'},
        'benign-update': {'what': 'admin installs updates, then sudo', 'expected_severity': 'Low',
                          'note': 'a mapper may suggest T1548.003 (sudo) with weak support; severity must stay '
                                  'Low/Medium at most: internal admin, no reconnaissance or exploitation before it'},
        'benign-av-refresh': {'what': 'antivirus full refresh (60 alerts in 1 min)', 'expected_severity': 'Low'}},
    'pass_condition_whole_system': 'no incident rated High or Critical; each report lists the innocent explanation',
    'expected_agent_v1': {
        'non_routine_incidents': 4,
        'benign-monitoring': 'routine, no incident (regular on day one)',
        'benign-backup': '1 incident per night (2) - known limitation: a once-a-day job is not learned from a 24 h baseline',
        'benign-update': '1 incident, update and sudo joined by a STRONG link',
        'benign-av-refresh': '1 incident of its own (spike), NOT joined to the update 3 min earlier (other machine)',
        'background': 'all routine, no incident'},
}

scn2_alerts, scn2_truth = build_scenario_2()
scn2_alerts.to_csv(SCN / 'scenario2_alerts.csv', index=False)
scn2_truth.to_csv(SCN / 'scenario2_truth.csv', index=False)
with open(SCN / 'scenario2_expected.json', 'w') as f:
    json.dump(expected_2, f, indent=2)

print(len(scn2_alerts), 'alerts written to', SCN.resolve())
scn2_truth['gt_incident'].value_counts()

1090 alerts written to /content/drive/MyDrive/sentinelai/scenarios


,count
gt_incident,
normal,480
benign-monitoring,480
benign-av-refresh,60
benign-backup-1,30
benign-backup-2,30
benign-update,10


In [26]:
# 1) the agent runs on the alerts only
a2, g2, l2, inc2 = agent(scn2_alerts)

# 2) EVALUATION: compare with the answers
t2 = a2.merge(scn2_truth, on='alert_id')
t2['kept'] = ~t2['incident_id'].isin(set(inc2.loc[inc2['routine'], 'incident_id']))

print('What the agent made of each true event:')
print(t2.groupby('gt_incident', sort=False)
        .agg(alerts=('alert_id', 'size'), kept=('kept', 'sum'),
             agent_incidents=('incident_id', lambda s: ', '.join(sorted(set(s))) if len(set(s)) <= 3 else f'{len(set(s))} routine'))
        .to_string())
print()


def incs2(gt):
    return set(t2.loc[(t2['gt_incident'] == gt) & t2['kept'], 'incident_id'])


kept_inc = t2[t2['kept']]
sudo_first = t2.loc[t2['short'] == 'W-Aut-Sudo', 'alert_id'].iloc[0]
checks2 = {
    'background is all routine': not t2.loc[t2['gt_incident'] == 'normal', 'kept'].any(),
    'monitoring scan (Nmap alert) is all routine': not t2.loc[t2['gt_incident'] == 'benign-monitoring', 'kept'].any(),
    '4 non-routine incidents': int((~inc2['routine']).sum()) == 4,
    'each backup night is ONE incident, nights separate': len(incs2('benign-backup-1')) == 1 and len(incs2('benign-backup-2')) == 1 and incs2('benign-backup-1') != incs2('benign-backup-2'),
    'update + sudo form ONE incident': len(incs2('benign-update')) == 1,
    'sudo joined by a STRONG link': bool((l2.loc[l2['alert_id'] == sudo_first, 'strength'] == 'strong').any()),
    'antivirus refresh is its own incident (not joined to the update)': len(incs2('benign-av-refresh')) == 1 and incs2('benign-av-refresh').isdisjoint(incs2('benign-update')),
    'no incident mixes two different true events': bool((kept_inc.groupby('incident_id')['gt_incident'].nunique() == 1).all()),
}
for name, ok in checks2.items():
    print('OK  ' if ok else 'DIFF', name)
print()
print(sum(checks2.values()), 'of', len(checks2), 'predictions met')

show2 = inc2[~inc2['routine']].copy()
show2['start'] = pd.to_datetime(show2['first_ts'], unit='s')
show2[['incident_id', 'hosts', 'groups', 'alerts', 'start', 'weak_links', 'confidence', 'types']]

What the agent made of each true event:
                   alerts  kept agent_incidents
gt_incident                                    
normal                480     0     480 routine
benign-monitoring     480     0      48 routine
benign-backup-1        30    30      inc-000023
benign-backup-2        30    30      inc-000288
benign-update          10    10      inc-000389
benign-av-refresh      60    60      inc-000390

OK   background is all routine
OK   monitoring scan (Nmap alert) is all routine
OK   4 non-routine incidents
OK   each backup night is ONE incident, nights separate
OK   update + sudo form ONE incident
OK   sudo joined by a STRONG link
OK   antivirus refresh is its own incident (not joined to the update)
OK   no incident mixes two different true events

8 of 8 predictions met


,incident_id,hosts,groups,alerts,start,weak_links,confidence,types
22,inc-000023,[internal_share],2,30,2026-01-20 02:00:00,0,high,[W-Sys-Big]
287,inc-000288,[internal_share],2,30,2026-01-21 02:00:00,0,high,[W-Sys-Big]
388,inc-000389,[webserver],2,10,2026-01-21 11:00:00,0,high,"[W-Aut-Sudo, W-Pkg-Ins]"
389,inc-000390,[mail],1,60,2026-01-21 11:03:00,0,low,[W-Sys-Cav]


## Stage 4 - Omar's mapper (O-3), Rules 4 and 5, and R-5 IncidentUpdates  *(24 Nov)*

**Checked against Omar's O-1/O-3 delivery.** It plugs into this agent, but on AIT-ADS alerts it first crashed, then mapped nothing. Three small fixes were made to **Omar's** code and sent back to him for review (`SentinelAI-O-3_v2_for_Omar.zip`, explained in `CHANGES_FOR_OMAR.md`):
1. alerts without `src_ip` (all AIT-ADS CSV alerts) no longer crash the mapper;
2. IDS scan signatures can now reach T1595 (an observation route like his other techniques);
3. privilege escalation added to O-1 as **T1548.003** (sourced from MITRE; no Kill Chain stage, because the Kill Chain has none).

**What this stage adds to my agent**
- **S4-0 / S4-1** take Omar's **original** zip and apply the three changes (plus the team contract's `host_ip` amendment) as visible patches, working in `/content` so nothing in Drive is touched. Fingerprints prove the result is byte-for-byte the v2 I am sending him.
- **S4-2 observation table:** which IDS alert types count as evidence for O-3. Chosen by what each signature *means*, never by which alerts fire during attacks. **Weak only:** an IDS signature alone never makes a technique strong. No AIT signature really shows a web shell, so T1505.003 stays unmapped on purpose.
- **S4-3 agent v2:** Rule 1 groups now keep their src/dst IPs (used for linking; the AIT CSV has none, so tuned results do not change). **Rule 4:** a step whose Kill Chain stage comes *before* a stage the incident already reached is still linked, but weak and marked `out_of_order`. **Rule 5 = decision (b):** a later incident joins an earlier one if they share a machine/IP, the gap is over 45 min but at most 24 h, and its earliest stage comes strictly *after* the latest stage already reached; always weak, marked `long_gap`.
- **S4-4 R-5:** contract `IncidentUpdate` messages; techniques and stages come from Omar's mapper. A stage is given only when **two different alert types** support the same step (the second one's log is the independent evidence Omar's rules require).

In [27]:
# ===== S4-0: Omar's delivery (SentinelAI-O-3.zip) into MyDrive/sentinelai - kept there after the first time =====
import os
import shutil
from pathlib import Path
print('working folder:', os.getcwd())
if not Path('SentinelAI-O-3.zip').exists():
    # first time only: upload his zip with the Files panel (left) -> Upload; it lands in /content
    shutil.copy('/content/SentinelAI-O-3.zip', 'SentinelAI-O-3.zip')
print('Omar zip in Drive:', Path('SentinelAI-O-3.zip').resolve(), Path('SentinelAI-O-3.zip').stat().st_size, 'bytes')

working folder: /content/drive/MyDrive/sentinelai
Omar zip in Drive: /content/drive/MyDrive/sentinelai/SentinelAI-O-3.zip 63356 bytes


In [28]:
# ===== S4-1: Omar's O-1 / O-3 (his zip) + my integration changes, applied visibly =====
# 1) Omar's original zip (cell above) is unpacked to /content/omar_work (outside Drive).
# 2) The changes below turn it into v2 - the same files as SentinelAI-O-3_v2_for_Omar.zip.
# 3) Fingerprints prove the result is byte-for-byte that v2.
import hashlib
from pathlib import Path
import json
import shutil
import sys
import zipfile

# work on a copy in the Colab machine (/content), never inside Drive - nothing in Drive is changed or deleted
WORK = Path('/content/omar_work')
shutil.rmtree(WORK, ignore_errors=True)
with zipfile.ZipFile('SentinelAI-O-3.zip') as z:
    z.extractall(WORK / 'original')
shutil.copytree(WORK / 'original/sentinelai', WORK / 'repo')    # repo = Omar's repository root
REPO = WORK / 'repo'


def patch(path, old, new):
    p = REPO / path
    text = p.read_text(encoding='utf-8')
    assert text.count(old) == 1, f'{path}: expected text not found - is this Omar\'s original?'
    p.write_text(text.replace(old, new), encoding='utf-8')


# --- change 1 (mapper.py): alerts without src_ip no longer crash the mapper ---
patch('sentinelai/frameworks/mapper.py',
'''        hits = set(rule["detection_rule_ids"]).intersection(alert["rule_hits"])
        source = ip_address(alert["src_ip"])
        if not hits or source in authorized:
            continue
        refs = _refs(alert, rule, hits)
        _record(techniques, "T1595", "weak", refs)
        if source in external:''',
'''        hits = set(rule["detection_rule_ids"]).intersection(alert["rule_hits"])
        # Integration fix (Reem, 24 Nov): check for a scan hit first, and skip alerts
        # with no src_ip. AIT-ADS IDS alerts from the CSV export have only host_ip
        # (contract amendment), and ip_address(None) used to crash the whole mapping.
        if not hits or alert.get("src_ip") is None:
            continue
        source = ip_address(alert["src_ip"])
        if source in authorized:
            continue
        refs = _refs(alert, rule, hits)
        _record(techniques, "T1595", "weak", refs)
        if source in external and alert.get("dst_ip") is not None:''')

# --- change 2 (mapper.py): IDS scan signatures can reach T1595 through an observation ---
patch('sentinelai/frameworks/mapper.py',
'''_OBSERVATION_RULES = {
    "T1190":''',
'''_OBSERVATION_RULES = {
    # Added 24 Nov (integration with the Investigation Agent): IDS scan signatures
    # (e.g. Suricata "ET SCAN ...") carry no detector rule hit, so they need an
    # observation route like the other techniques.
    "T1595": ("active_scanning_suspected", "active_scanning_verified",
              "target_discovery", "reconnaissance"),
    "T1190":''')

# --- change 3 (mapper.py + O-1 JSON): privilege escalation T1548.003 ---
patch('sentinelai/frameworks/mapper.py',
'''              "data_theft_objective", "actions_on_objectives"),
}''',
'''              "data_theft_objective", "actions_on_objectives"),
    # Added 24 Nov: privilege escalation via sudo (AIT attack phase). The Kill Chain
    # has no privilege-escalation stage, so O-1 gives this rule no stage hypothesis
    # and the stage observation below can never produce one.
    "T1548.003": ("sudo_abuse_suspected", "sudo_abuse_verified",
                  "sudo_abuse_stage", "installation"),
}''')
p = REPO / 'sentinelai/frameworks/attack_techniques.json'
d = json.loads(p.read_text(encoding='utf-8'))
d['techniques'].append({
    'technique_id': 'T1548.003', 'name': 'Abuse Elevation Control Mechanism: Sudo and Sudo Caching',
    'tactics': ['Privilege Escalation'], 'source_version': '2.0',
    'description': 'Use of sudo, its cached credentials or a permissive sudoers configuration to run commands with higher privileges.',
    'source_url': 'https://attack.mitre.org/techniques/T1548/003/', 'parent_technique_id': 'T1548'})
p.write_text(json.dumps(d, indent=2, ensure_ascii=False) + chr(10), encoding='utf-8')
p = REPO / 'sentinelai/frameworks/technique_stage_mapping.json'
d = json.loads(p.read_text(encoding='utf-8'))
d['rules'].append({
    'mapping_rule_id': 'O1-T1548.003', 'technique_id': 'T1548.003', 'detection_rule_ids': [],
    'technique_evidence': {
        'weak': 'Host authentication or audit evidence of sudo to root on a host involved in the incident.',
        'strong': ('Linked host evidence that the elevation was not routine administration, such as an unexpected '
                   'user, a first-time sudo, or elevation following other attack activity on the same host.'),
        'limitations': 'Administrators use sudo routinely; elevation alone does not establish abuse.'},
    'stage_hypotheses': []})
p.write_text(json.dumps(d, indent=2, ensure_ascii=False) + chr(10), encoding='utf-8')

# --- team contract amendment (messages.py): host_ip; IDS alerts may lack src/dst but then need host_ip ---
patch('sentinelai/contracts/messages.py',
'''    _require(m, ("network", "detector", "first_seen", "last_seen", "src_ip", "dst_ip",''',
'''    _require(m, ("network", "detector", "first_seen", "last_seen", "host_ip", "src_ip", "dst_ip",''')
patch('sentinelai/contracts/messages.py',
'''    for k in ("src_ip", "dst_ip"):
        if not _is_ip(m[k]):
            e.append(f"{w}: {k} is not a valid IP (IPs are required, never dropped)")''',
'''    # IPs are never dropped. Detection Agent alerts always come from a flow, so
    # src_ip and dst_ip are required. IDS alerts (AIT-ADS) keep every IP the
    # source record has; when it has no src/dst (e.g. the CSV export), the IP
    # of the host that raised the alert (host_ip) is required instead.
    for k in ("src_ip", "dst_ip", "host_ip"):
        if m[k] is not None and not _is_ip(m[k]):
            e.append(f"{w}: {k} is not a valid IP")
    if m["detector"] == "ids":
        if m["host_ip"] is None and (m["src_ip"] is None or m["dst_ip"] is None):
            e.append(f"{w}: ids alerts need host_ip, or both src_ip and dst_ip")
    else:
        for k in ("src_ip", "dst_ip"):
            if m[k] is None:
                e.append(f"{w}: {k} is required (IPs are never dropped)")''')

# --- prove the result is exactly v2 ---
V2 = {'sentinelai/frameworks/mapper.py': '1f0e37d297f3fc7a', 'sentinelai/contracts/messages.py': '2bd8de1b3824dd37',
      'sentinelai/frameworks/attack_techniques.json': '06136681a87c3311', 'sentinelai/frameworks/technique_stage_mapping.json': 'fbaac4a39ea4ef16'}
for path, expected in V2.items():
    got = hashlib.sha256((REPO / path).read_bytes()).hexdigest()[:16]
    print('OK  ' if got == expected else 'DIFF', path, got)
assert all(hashlib.sha256((REPO / p).read_bytes()).hexdigest()[:16] == h for p, h in V2.items()), 'not identical to v2'

sys.path.insert(0, str(REPO.resolve()))
for m in [m for m in sys.modules if m == 'sentinelai' or m.startswith('sentinelai.')]:
    del sys.modules[m]
from sentinelai.contracts.messages import KILL_CHAIN_STAGES, validate_bundle, validate_message
from sentinelai.frameworks.mapper import EvidenceObservation, map_alerts

O1_RULES = json.loads((REPO / 'sentinelai/frameworks/technique_stage_mapping.json').read_text())['rules']
print('O-1 techniques:', [r['technique_id'] for r in O1_RULES])


OK   sentinelai/frameworks/mapper.py 1f0e37d297f3fc7a
OK   sentinelai/contracts/messages.py 2bd8de1b3824dd37
OK   sentinelai/frameworks/attack_techniques.json 06136681a87c3311
OK   sentinelai/frameworks/technique_stage_mapping.json fbaac4a39ea4ef16
O-1 techniques: ['T1595', 'T1190', 'T1505.003', 'T1059', 'T1048', 'T1548.003']


In [29]:
# ===== S4-2: observation table (which IDS alerts count as evidence) + stage lookup =====
# Chosen by what each signature MEANS (read from its text), never by which alerts fire during attacks.
# Only 'suspected' (= weak) kinds: an IDS signature alone never makes a technique strong.
OBSERVATION_RULES = {
    # T1595 Active Scanning
    'Suricata: Alert - ET SCAN Possible Nmap User-Agent Observed': 'active_scanning_suspected',
    'Suricata: Alert - ET SCAN Behavioral Unusual Port 445 traffic Potential Scan or Infection': 'active_scanning_suspected',
    'Wazuh: sshd: insecure connection attempt (scan).': 'active_scanning_suspected',
    'Wazuh: Multiple web server 400 error codes from same source ip.': 'active_scanning_suspected',
    # T1190 Exploit Public-Facing Application
    'Wazuh: Common web attack.': 'public_app_exploit_suspected',
    # T1059 Command and Scripting Interpreter
    'AMiner: New user_cmd parameter combination in Audit logs.': 'interpreter_suspected',
    # T1548.003 Sudo and Sudo Caching
    'Wazuh: Successful sudo to ROOT executed.': 'sudo_abuse_suspected',
    'Wazuh: First time user executed sudo.': 'sudo_abuse_suspected',
    # T1048 Exfiltration Over Alternative Protocol (data hidden in DNS names)
    'AMiner: High entropy in DNS domain.': 'alternative_exfiltration_suspected',
    'AMiner: New characters in DNS domain.': 'alternative_exfiltration_suspected',
}
# Not mapped on purpose: no AIT signature specifically shows a web shell (T1505.003).
KIND_TECHNIQUE = {'active_scanning_suspected': 'T1595', 'public_app_exploit_suspected': 'T1190',
                  'interpreter_suspected': 'T1059', 'sudo_abuse_suspected': 'T1548.003',
                  'alternative_exfiltration_suspected': 'T1048'}
STAGE_KIND = {'T1595': 'target_discovery', 'T1190': 'exploitation_stage',
              'T1059': 'remote_tasking', 'T1048': 'data_theft_objective'}   # T1548.003: no Kill Chain stage
STAGE_NO = {stage: i for i, stage in enumerate(KILL_CHAIN_STAGES, start=1)}
POSSIBLE_STAGE = {r['technique_id']: [h['stage'] for h in r['stage_hypotheses']] for r in O1_RULES}


def signature_stage(signature):
    """Kill Chain position (1-7) a signature may point to, from O-1; None if unknown."""
    stages = POSSIBLE_STAGE.get(KIND_TECHNIQUE.get(OBSERVATION_RULES.get(signature)), [])
    return min(STAGE_NO[s] for s in stages) if stages else None


In [30]:
# ===== S4-3: agent v2 - Rule 1 keeps every IP, Rule 4 order check, Rule 5 long-gap pass =====
def _ips(s):
    return sorted({ip for ip in s.dropna()})


def merge_bursts(alerts, burst_gap_s=60, max_group_span_s=600):
    """Rule 1 (v2: same grouping as v1, and each group now keeps its src/dst IPs and signatures)."""
    a = alerts.sort_values(['host', 'short', 'ts', 'alert_id'], kind='stable').copy()
    for col in ('src_ip', 'dst_ip'):
        if col not in a:
            a[col] = None
    host, short = a['host'].astype(object), a['short'].astype(object)   # same result on any pandas version
    same_type = ((host == host.shift()) & (short == short.shift())).fillna(False).astype(bool)
    new_run = ~same_type | (a['ts'].diff() > burst_gap_s).fillna(False).astype(bool)
    run_id = new_run.astype('int64').cumsum()
    run_start = a.groupby(run_id)['ts'].transform('min')
    piece = (a['ts'] - run_start) // max_group_span_s
    a['group_key'] = run_id.astype(str) + '-' + piece.astype(str)
    groups = (a.groupby('group_key', sort=False)
                .agg(host=('host', 'first'), host_ip=('host_ip', 'first'),
                     short=('short', 'first'), signature=('signature', 'first'),
                     signatures=('signature', lambda s: sorted(set(s))),
                     src_ips=('src_ip', _ips), dst_ips=('dst_ip', _ips),
                     first_ts=('ts', 'min'), last_ts=('ts', 'max'),
                     count=('ts', 'size'), first_alert=('alert_id', 'first'))
                .sort_values('first_ts', kind='stable')
                .reset_index())
    groups['group_id'] = [f'grp-{i:06d}' for i in range(len(groups))]
    key_to_id = dict(zip(groups['group_key'], groups['group_id']))
    group_of_alert = a['group_key'].map(key_to_id).reindex(alerts.index)
    return groups.drop(columns='group_key'), group_of_alert


def entities(group):
    ents = {f"host:{group['host']}"}
    if isinstance(group.get('host_ip'), str):
        ents.add(f"ip:{group['host_ip']}")
    for col in ('src_ips', 'dst_ips'):
        for ip in group.get(col) or []:
            ents.add(f'ip:{ip}')
    return ents


def group_stage(group):
    stages = [signature_stage(s) for s in group.get('signatures') or [group.get('signature')]]
    stages = [s for s in stages if s]
    return min(stages) if stages else None


def link_groups(groups, max_gap_s=2700, strong_gap_s=300):
    """Rule 3 + Rule 4. Linking is unchanged from v1 (shared machine/IP within max_gap_s).
    Rule 4: if a step's Kill Chain stage comes BEFORE a stage the incident already reached,
    the link is kept but marked weak and 'out_of_order' (never dropped silently)."""
    groups = groups.sort_values(['first_ts', 'group_id'], kind='stable').reset_index(drop=True)
    incident_of, links = [], []
    max_stage = {}      # incident -> latest Kill Chain stage reached so far
    last_seen = {}      # entity -> {'incident_id', 'last_ts', 'alert_id'}
    n = 0
    for row in groups.to_dict('records'):
        if row['routine']:
            n += 1
            incident_of.append(f'inc-{n:06d}')
            continue
        ents, stage, best = entities(row), group_stage(row), None
        for e in sorted(ents):
            seen = last_seen.get(e)
            if seen is None:
                continue
            gap = row['first_ts'] - seen['last_ts']
            if gap <= max_gap_s and (best is None or gap < best[0]):
                best = (gap, e, seen)
        if best is None:
            n += 1
            iid = f'inc-{n:06d}'
        else:
            gap, e, seen = best
            iid = seen['incident_id']
            reached = max_stage.get(iid)
            if stage is None or reached is None:
                order = 'unknown'
            else:
                order = 'out_of_order' if stage < reached else 'in_order'
            links.append({'incident_id': iid, 'alert_id': row['first_alert'],
                          'linked_to': seen['alert_id'], 'shared_entity': e,
                          'gap_seconds': int(max(gap, 0)),
                          'strength': 'strong' if gap <= strong_gap_s and order != 'out_of_order' else 'weak',
                          'order': order, 'long_gap': False})
        if stage is not None:
            max_stage[iid] = max(max_stage.get(iid, 0), stage)
        incident_of.append(iid)
        for e in ents:
            prev = last_seen.get(e)
            if prev is None or prev['incident_id'] != iid or row['last_ts'] >= prev['last_ts']:
                last_seen[e] = {'incident_id': iid, 'last_ts': row['last_ts'], 'alert_id': row['first_alert']}
    groups['incident_id'] = incident_of
    links = pd.DataFrame(links, columns=['incident_id', 'alert_id', 'linked_to', 'shared_entity',
                                         'gap_seconds', 'strength', 'order', 'long_gap'])
    return groups, links


def join_long_gaps(groups, links, max_gap_s=2700, max_long_gap_s=24 * 3600):
    """Rule 5 - decision (b). A later incident B joins an earlier incident A when:
    they share a machine or IP, B starts more than max_gap_s but at most max_long_gap_s after A ends,
    and B's earliest Kill Chain stage comes strictly AFTER the latest stage A reached.
    Such a join is always WEAK and marked long_gap. Incidents without a known stage never join."""
    g = groups.copy()
    g['stage'] = [group_stage(r) for r in g.to_dict('records')]
    real = g[~g['routine']]
    inc = {}
    for iid, part in real.groupby('incident_id', sort=False):
        rows = part.sort_values('first_ts').to_dict('records')
        stages = [r['stage'] for r in rows if r['stage']]
        inc[iid] = {'first_ts': rows[0]['first_ts'], 'last_ts': max(r['last_ts'] for r in rows),
                    'ents': set().union(*[entities(r) for r in rows]),
                    'min_stage': min(stages) if stages else None, 'max_stage': max(stages) if stages else None,
                    'first_alert': rows[0]['first_alert'], 'last_alert': rows[-1]['first_alert']}
    merged_into, new_links = {}, []
    for b in sorted(inc, key=lambda k: inc[k]['first_ts']):
        B = inc[b]
        if B['min_stage'] is None:
            continue
        best = None
        for a_id, A in inc.items():
            if a_id == b or a_id in merged_into or A['max_stage'] is None:
                continue
            gap = B['first_ts'] - A['last_ts']
            shared = sorted(A['ents'] & B['ents'])
            if max_gap_s < gap <= max_long_gap_s and shared and B['min_stage'] > A['max_stage']:
                if best is None or gap < best[0]:
                    best = (gap, a_id, shared[0])
        if best:
            gap, a_id, e = best
            A = inc[a_id]
            new_links.append({'incident_id': a_id, 'alert_id': B['first_alert'], 'linked_to': A['last_alert'],
                              'shared_entity': e, 'gap_seconds': int(gap), 'strength': 'weak',
                              'order': 'in_order', 'long_gap': True})
            merged_into[b] = a_id
            A['last_ts'] = max(A['last_ts'], B['last_ts'])
            A['ents'] |= B['ents']
            A['max_stage'] = max(A['max_stage'], B['max_stage'])
            A['last_alert'] = B['last_alert']
    if merged_into:
        g['incident_id'] = g['incident_id'].replace(merged_into)
        links = links.assign(incident_id=links['incident_id'].replace(merged_into))
        links = pd.concat([links, pd.DataFrame(new_links)], ignore_index=True)
    return g.drop(columns='stage'), links


def summarise_incidents(groups, links):
    weak = links[links['strength'] == 'weak'].groupby('incident_id').size()
    inc = groups.groupby('incident_id').agg(
        routine=('routine', 'all'), groups=('group_id', 'size'), alerts=('count', 'sum'),
        hosts=('host', lambda s: sorted(set(s))), first_ts=('first_ts', 'min'),
        last_ts=('last_ts', 'max'), types=('short', lambda s: sorted(set(s)))).reset_index()
    inc['weak_links'] = inc['incident_id'].map(weak).fillna(0).astype(int)
    inc['confidence'] = np.select([inc['routine'] | (inc['groups'] == 1), inc['weak_links'] > 0],
                                  ['low', 'medium'], default='high')
    return inc


def agent(a, cfg=CFG):
    """The Investigation Agent v2 (Rules 1-5). Reads only the alerts table."""
    g, g_of_a = merge_bursts(a, cfg['burst_gap_s'], cfg['max_group_span_s'])
    g = mark_routine(g, routine_baseline(a, cfg['baseline_hours']), cfg['routine_min_bins'], cfg['spike_factor'])
    g, links = link_groups(g, cfg['max_gap_s'], cfg['strong_gap_s'])
    g, links = join_long_gaps(g, links, cfg['max_gap_s'], cfg['max_long_gap_s'])
    inc = summarise_incidents(g, links)
    a = a.assign(incident_id=g_of_a.map(g.set_index('group_id')['incident_id']).values)
    return a, g, links, inc


CFG['max_long_gap_s'] = 24 * 3600     # Rule 5 limit (design choice, not tuned)


In [31]:
# ===== S4-4: R-5 - IncidentUpdate messages, techniques and stages from Omar's mapper =====
def _iso(ts):
    return pd.Timestamp(int(ts), unit='s', tz='UTC').strftime('%Y-%m-%dT%H:%M:%SZ')


def _val(x):
    return None if x is None or (isinstance(x, float) and np.isnan(x)) or x is pd.NA else x


def alert_message(r):
    """One row of the alerts table as a contract Alert message (IDS alert, every IP kept)."""
    return {'message_id': r['alert_id'], 'message_type': 'Alert', 'schema_version': '1.0',
            'created_at': r['time'], 'producer': 'ait_ids_loader',
            'evidence_refs': [f"raw_log:{r['network']}/{r['alert_id']}"],
            'network': r['network'], 'detector': 'ids', 'first_seen': r['time'], 'last_seen': r['time'],
            'host_ip': _val(r['host_ip']), 'src_ip': _val(r.get('src_ip')), 'dst_ip': _val(r.get('dst_ip')),
            'src_port': None, 'dst_port': None, 'protocol': None, 'rf_score': None, 'rule_hits': [],
            'ids_source': r['ids_source'], 'signature': r['signature'], 'user': None,
            'hostname': r['host'], 'summary': f"{r['short']}: {r['signature']}"}


def observations_for(sub):
    """Technique evidence from the table; stage evidence only when ANOTHER alert type in the
    same incident supports the same technique (its log is the independent evidence)."""
    obs = []
    rows = sub[sub['signature'].isin(list(OBSERVATION_RULES))].to_dict('records')
    first_by_sig = {}
    for r in rows:
        first_by_sig.setdefault(r['signature'], r)
    for r in rows:
        kind = OBSERVATION_RULES[r['signature']]
        own = f"raw_log:{r['network']}/{r['alert_id']}"
        obs.append(EvidenceObservation(kind, r['alert_id'], own))
        tech = KIND_TECHNIQUE[kind]
        other = next((o for s, o in first_by_sig.items()
                      if s != r['signature'] and KIND_TECHNIQUE[OBSERVATION_RULES[s]] == tech), None)
        if other is not None and tech in STAGE_KIND:
            obs.append(EvidenceObservation(STAGE_KIND[tech], r['alert_id'],
                                           f"raw_log:{other['network']}/{other['alert_id']}"))
    return obs


def incident_updates(a, g, links, inc, network, external_sources=()):
    msgs = []
    by_inc = dict(tuple(a.groupby('incident_id')))
    groups_by_inc = dict(tuple(g.groupby('incident_id')))
    links_by_inc = dict(tuple(links.groupby('incident_id')))
    for n, r in enumerate(inc[~inc['routine']].sort_values('first_ts').itertuples(index=False), start=1):
        sub = by_inc[r.incident_id].sort_values('ts', kind='stable')
        mapping = map_alerts([alert_message(x) for x in sub.to_dict('records')],
                             external_sources=external_sources, observations=observations_for(sub))
        lr = links_by_inc[r.incident_id].to_dict('records') if r.incident_id in links_by_inc else []
        ips = sorted({ip for col in ('host_ip', 'src_ip', 'dst_ip') if col in sub for ip in sub[col].dropna()})
        notes = []
        mapped = {t['technique_id'] for t in mapping['candidate_techniques']}
        if not mapped:
            notes.append('No ATT&CK technique supported by the available evidence.')
        elif all(t['support'] == 'weak' for t in mapping['candidate_techniques']):
            notes.append('All techniques are weak: IDS signatures alone, no reviewed host evidence.')
        if not mapping['stage_hypotheses']:
            notes.append('No Kill Chain stage hypothesis (needs two independent alert types per step).')
        if any(l['order'] == 'out_of_order' for l in lr):
            notes.append(f"{sum(l['order'] == 'out_of_order' for l in lr)} link(s) out of Kill Chain order (kept, marked weak).")
        if any(l['long_gap'] for l in lr):
            notes.append('Joined across a long gap by shared machine/IP and Kill Chain order (Rule 5, weak).')
        if r.weak_links:
            notes.append(f'{r.weak_links} weak link(s) in total.')
        if r.groups == 1:
            notes.append('Single alert group; nothing else linked to it.')
        firsts = groups_by_inc[r.incident_id].sort_values('first_ts')['first_alert'].tolist()
        msgs.append({
            'message_id': f'incupd-{network}-{n:06d}', 'message_type': 'IncidentUpdate',
            'schema_version': '1.0', 'created_at': _iso(r.last_ts), 'producer': 'investigation_agent',
            'evidence_refs': [f'alert:{x}' for x in firsts],
            'incident_id': r.incident_id.replace('inc-', f'inc-{network}-', 1),
            'version': 1, 'status': 'open', 'network': network,
            'alert_ids': sub['alert_id'].tolist(),
            'entities': {'ips': ips, 'hosts': list(r.hosts), 'users': []},
            'start_time': _iso(r.first_ts), 'end_time': _iso(r.last_ts),
            'link_reasons': [{k: (int(l[k]) if k == 'gap_seconds' else (bool(l[k]) if k == 'long_gap' else l[k]))
                              for k in ('alert_id', 'linked_to', 'shared_entity', 'gap_seconds',
                                        'strength', 'order', 'long_gap')} for l in lr],
            'candidate_techniques': mapping['candidate_techniques'],
            'stage_hypotheses': mapping['stage_hypotheses'],
            'confidence': r.confidence, 'uncertainty_notes': ' '.join(notes),
        })
    return msgs


### Step 4 tests - 13 new tests (Rules 1, 4, 5 and R-5) + the 16 from Step 3.1, all run against agent v2

In [32]:
# ===== CELL S4-T: unit tests for Rule 1 (IPs), Rule 4, Rule 5 and R-5 =====
IPS.update({'dns': '10.0.0.4'})
TEST_CFG2 = {**CFG, 'baseline_hours': 1, 'routine_min_bins': 3}
SIGS = {'scan': 'Wazuh: Multiple web server 400 error codes from same source ip.',   # T1595, stage 1
        'nmap': 'Suricata: Alert - ET SCAN Possible Nmap User-Agent Observed',      # T1595, stage 1
        'exploit': 'Wazuh: Common web attack.',                                     # T1190, stage 4
        'cmd': 'AMiner: New user_cmd parameter combination in Audit logs.',         # T1059, stage 6
        'sudo': 'Wazuh: Successful sudo to ROOT executed.',                         # T1548.003, no stage
        'dns1': 'AMiner: High entropy in DNS domain.',                              # T1048, stage 7
        'dns2': 'AMiner: New characters in DNS domain.',                            # T1048, stage 7
        'plain': 'test: unmapped alert'}


def make_alerts2(rows):
    """rows = (seconds after T0, machine, signature key, src_ip)"""
    a = pd.DataFrame(rows, columns=['dt', 'host', 'sig', 'src_ip'])
    a['ts'] = T0 + a['dt']
    a = a.sort_values('ts', kind='stable').reset_index(drop=True)
    a['alert_id'] = [f'alert-t-{i:05d}' for i in range(len(a))]
    a['signature'] = a['sig'].map(SIGS)
    a['short'] = a['sig']
    a['host_ip'] = a['host'].map(IPS)
    a['dst_ip'] = None
    a['network'] = 't'
    a['ids_source'] = 'wazuh'
    a['time'] = pd.to_datetime(a['ts'], unit='s', utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    return a.drop(columns=['dt', 'sig'])


NORMAL_DAY2 = [(i * 600, 'mail', 'plain', None) for i in range(6)]


def run2(rows):
    return agent(make_alerts2(NORMAL_DAY2 + rows), TEST_CFG2)


def inc2(a, key):
    return a.loc[a['signature'] == SIGS[key], 'incident_id'].iloc[0]


def test_rule1_groups_keep_src_and_dst_ips():
    a = make_alerts2([(H, 'web', 'scan', '203.0.113.66')])
    a.loc[0, 'dst_ip'] = '10.0.0.1'
    g, _ = merge_bursts(a)
    assert g['src_ips'].iloc[0] == ['203.0.113.66'] and g['dst_ips'].iloc[0] == ['10.0.0.1']


def test_rule4_out_of_order_link_is_kept_but_weak():
    a, g, links, _ = run2([(H, 'web', 'exploit', None), (H + 60, 'web', 'scan', None)])   # scan AFTER exploit
    assert inc2(a, 'exploit') == inc2(a, 'scan'), 'out-of-order step was dropped'
    assert list(links['order']) == ['out_of_order'] and list(links['strength']) == ['weak']


def test_rule4_in_order_link_stays_strong():
    a, g, links, _ = run2([(H, 'web', 'scan', None), (H + 60, 'web', 'exploit', None)])
    assert list(links['order']) == ['in_order'] and list(links['strength']) == ['strong']


def test_rule4_unknown_stage_does_not_weaken():
    a, g, links, _ = run2([(H, 'web', 'exploit', None), (H + 60, 'web', 'sudo', None)])   # sudo has no stage
    assert list(links['order']) == ['unknown'] and list(links['strength']) == ['strong']


def test_rule5_joins_later_stage_across_long_gap():
    rows = [(H, 'web', 'exploit', None), (H + 60, 'web', 'cmd', None),
            (H + 5 * 3600, 'fw', 'dns1', IPS['web'])]           # 5 h later, from the web server's IP
    a, g, links, _ = run2(rows)
    assert inc2(a, 'dns1') == inc2(a, 'cmd'), 'long-gap step not joined'
    joined = links[links['long_gap']]
    assert len(joined) == 1 and joined['strength'].iloc[0] == 'weak'
    assert joined['shared_entity'].iloc[0] == f"ip:{IPS['web']}"


def test_rule5_never_joins_an_earlier_stage():
    rows = [(H, 'web', 'cmd', None), (H + 5 * 3600, 'fw', 'nmap', IPS['web'])]           # scan after C2
    a, g, links, _ = run2(rows)
    assert inc2(a, 'nmap') != inc2(a, 'cmd') and not links['long_gap'].any()


def test_rule5_needs_a_shared_machine_or_ip():
    rows = [(H, 'web', 'cmd', None), (H + 5 * 3600, 'fw', 'dns1', None)]
    a, g, links, _ = run2(rows)
    assert inc2(a, 'dns1') != inc2(a, 'cmd')


def test_rule5_never_joins_steps_without_a_stage():
    rows = [(H, 'web', 'plain', None), (H + 5 * 3600, 'web', 'plain', None)]
    a, g, links, _ = run2(rows)
    assert a.loc[a['ts'] >= T0 + H, 'incident_id'].nunique() == 2


def test_rule5_respects_the_24h_limit():
    rows = [(H, 'web', 'cmd', None), (H + 25 * 3600, 'fw', 'dns1', IPS['web'])]
    a, g, links, _ = run2(rows)
    assert inc2(a, 'dns1') != inc2(a, 'cmd')


def test_observation_table_only_uses_weak_kinds():
    assert all(kind.endswith('_suspected') for kind in OBSERVATION_RULES.values())
    assert set(KIND_TECHNIQUE.values()) <= {r['technique_id'] for r in O1_RULES}


def test_r5_incident_update_follows_contract_and_cites_evidence():
    a, g, links, inc = run2([(H, 'web', 'cmd', None), (H + 60, 'web', 'sudo', None)])
    msgs = incident_updates(a, g, links, inc, 't')
    m = next(x for x in msgs if len(x['alert_ids']) == 2)
    techs = {t['technique_id']: t for t in m['candidate_techniques']}
    assert set(techs) == {'T1059', 'T1548.003'} and all(t['support'] == 'weak' for t in techs.values())
    assert 'rule:O1-T1059' in techs['T1059']['evidence_refs']
    assert m['incident_id'].startswith('inc-t-')
    assert validate_message(m) == []
    alert_msgs = [alert_message(r) for r in a.to_dict('records') if r['alert_id'] in set(m['alert_ids'])]
    assert validate_bundle(alert_msgs + [m]) == [], validate_bundle(alert_msgs + [m])


def test_r5_stage_needs_two_independent_alert_types():
    one = incident_updates(*run2([(H, 'dns', 'dns1', None)]), 't')
    assert [s['stage'] for m in one for s in m['stage_hypotheses']] == []
    two = incident_updates(*run2([(H, 'dns', 'dns1', None), (H + 30, 'dns', 'dns2', None)]), 't')
    m = next(x for x in two if len(x['alert_ids']) == 2)
    assert [s['stage'] for s in m['stage_hypotheses']] == ['actions_on_objectives']


def test_r5_innocent_alerts_map_to_nothing():
    msgs = incident_updates(*run2([(H, 'web', 'plain', None)]), 't')
    assert all(m['candidate_techniques'] == [] for m in msgs)


# ---------- run ALL unit tests: the 16 from Step 3.1 + the 13 above ----------
tests = [(name, f) for name, f in list(globals().items()) if name.startswith('test_') and callable(f)]
passed = 0
for name, f in tests:
    try:
        f()
        print('PASS ', name)
        passed += 1
    except AssertionError as e:
        print('FAIL ', name, '->', e)
    except Exception as e:
        print('ERROR', name, '->', type(e).__name__, e)
print()
print(f'{passed} of {len(tests)} tests passed')


PASS  test_loader_keeps_answers_out
PASS  test_repeats_within_60s_merge
PASS  test_gap_over_60s_splits
PASS  test_endless_stream_is_capped_at_10_min
PASS  test_different_types_not_merged
PASS  test_regular_type_at_normal_volume_is_routine
PASS  test_spike_of_regular_type_is_kept
PASS  test_type_never_seen_is_kept
PASS  test_same_machine_2_min_apart_linked_strong
PASS  test_different_machines_5s_apart_stay_separate
PASS  test_gap_of_exactly_45_min_links
PASS  test_gap_over_45_min_stays_separate
PASS  test_10_min_gap_is_a_weak_link
PASS  test_routine_alert_never_links
PASS  test_shared_attacker_ip_links_two_machines
PASS  test_confidence_levels
PASS  test_rule1_groups_keep_src_and_dst_ips
PASS  test_rule4_out_of_order_link_is_kept_but_weak
PASS  test_rule4_in_order_link_stays_strong
PASS  test_rule4_unknown_stage_does_not_weaken
PASS  test_rule5_joins_later_stage_across_long_gap
PASS  test_rule5_never_joins_an_earlier_stage
PASS  test_rule5_needs_a_shared_machine_or_ip
PASS  test_rule5_n

### Scenarios v2 - the same two stories, now with **real AIT-ADS signatures** (so the observation table applies exactly as on real data)

The files in `scenarios/` are replaced (v2). **Predictions written before running:**
- **Scenario 1:** 2 incidents. The **whole attack is ONE incident**: the firewall scan joins through the shared attacker IP (Rule 3), privilege escalation by a weak link, the DNS theft 6 h later by a weak long-gap link (Rule 5). The typo stays alone. Techniques, all weak: T1048, T1059, T1190, T1548.003, T1595. Stages: reconnaissance and actions on objectives (the two steps with two independent alert types). T1505.003 not mapped.
- **Scenario 2:** still 4 incidents; the update + sudo incident gets only weak T1548.003 (innocent admin - must end up Low); no stages; no long-gap joins.

In [33]:
# ===== S4-5: scenarios v2 - same stories, real AIT-ADS signatures =====
SCN = Path('scenarios')
SCN.mkdir(exist_ok=True)
ATTACKER = '203.0.113.66'
MACHINE_IP = {'webserver': '10.0.1.20', 'inet-firewall': '10.0.1.1', 'mail': '10.0.1.25',
              'inet-dns': '10.0.1.53', 'vpn': '10.0.1.10', 'monitoring': '10.0.1.5',
              'internal_share': '10.0.1.40', 'backup': '10.0.1.30'}
SIG = {   # real AIT-ADS signatures used by the scenarios
    'clamav': ('wazuh', 'W-Sys-Cav', 'Wazuh: ClamAV database update'),
    'ids_evt': ('wazuh', 'W-All-Evt', 'Wazuh: IDS event.'),
    'nmap': ('suricata', 'S-Flw-Nmp', 'Suricata: Alert - ET SCAN Possible Nmap User-Agent Observed'),
    'mail_inv': ('wazuh', 'W-Mai-Inv', 'Wazuh: Dovecot Invalid User Login Attempt.'),
    'web400': ('wazuh', 'W-Acc-400', 'Wazuh: Web server 400 error code.'),
    'web400_multi': ('wazuh', 'W-All-Mul3', 'Wazuh: Multiple web server 400 error codes from same source ip.'),
    'new_method': ('aminer', 'A-Acc-Val1', 'AMiner: New request method in Apache Access log.'),
    'forbidden': ('wazuh', 'W-Err-Fbd2', 'Wazuh: Apache: Attempt to access forbidden file or directory.'),
    'web_attack': ('wazuh', 'W-Acc-Att', 'Wazuh: Common web attack.'),
    'suspicious_url': ('wazuh', 'W-Acc-Sus', 'Wazuh: Suspicious URL access.'),
    'user_cmd': ('aminer', 'A-Aud-Com6', 'AMiner: New user_cmd parameter combination in Audit logs.'),
    'pam_open': ('wazuh', 'W-Aut-Pam1', 'Wazuh: PAM: Login session opened.'),
    'sudo_first': ('wazuh', 'W-Aut-Sud', 'Wazuh: First time user executed sudo.'),
    'sudo_root': ('wazuh', 'W-Aut-Sud', 'Wazuh: Successful sudo to ROOT executed.'),
    'dns_entropy': ('aminer', 'A-Dns-Ent', 'AMiner: High entropy in DNS domain.'),
    'dns_chars': ('aminer', 'A-Dns-Chr', 'AMiner: New characters in DNS domain.'),
    'apt': ('suricata', 'S-Flw-Apt', 'Suricata: Alert - ET POLICY GNU/Linux APT User-Agent Outbound likely related to package management'),
    'new_event': ('aminer', 'A-All-Evt', 'AMiner: New event type.'),
}


def _finish(rows, name, day0):
    t0 = int(pd.Timestamp(day0, tz='UTC').timestamp())
    df = pd.DataFrame(rows)
    df['ts'] = t0 + df['dt']
    df = df.sort_values('ts', kind='stable').reset_index(drop=True)
    df['alert_id'] = [f'alert-{name}-{i:05d}' for i in range(len(df))]
    df['network'] = name
    df['time'] = pd.to_datetime(df['ts'], unit='s', utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    df['host_ip'] = df['host'].map(MACHINE_IP)
    agent_cols = ['alert_id', 'network', 'ts', 'time', 'ids_source', 'signature', 'short',
                  'host', 'host_ip', 'src_ip', 'dst_ip']
    truth_cols = ['alert_id', 'gt_incident', 'gt_step', 'gt_technique', 'gt_stage']
    return df[agent_cols], df[truth_cols]


def _adder(rows):
    def add(dt, host, sig, src_ip=None, dst_ip=None, gt='normal', step='normal', technique=None, stage=None):
        tool, short, signature = SIG[sig]
        rows.append(dict(dt=dt, host=host, ids_source=tool, short=short, signature=signature,
                         src_ip=src_ip, dst_ip=dst_ip, gt_incident=gt, gt_step=step,
                         gt_technique=technique, gt_stage=stage))
    return add


def _background(add):
    for t in range(0, 2 * 86400, 600):
        add(t, 'mail', 'clamav')
    for t in range(300, 2 * 86400, 1800):
        add(t, 'vpn', 'ids_evt')
        add(t + 5, 'inet-firewall', 'ids_evt')


def build_scenario_1():
    """Web attack. Day 1 normal; day 2 10:00 attacker 203.0.113.66."""
    rows = []
    add = _adder(rows)
    _background(add)
    A, web = 86400 + 10 * 3600, MACHINE_IP['webserver']
    for i in range(20):                                   # 1 network scan (firewall sees it)
        add(A + 2 * i, 'inet-firewall', 'nmap', ATTACKER, f'10.0.1.{1 + i}',
            'GT-1', 'network_scan', 'T1595', 'reconnaissance')
    add(A + 5, 'mail', 'mail_inv', '10.0.2.15', MACHINE_IP['mail'], 'benign-1', 'typo_login')   # coincidence
    for i in range(50):                                   # 2 service scan of the web server
        add(A + 120 + i, 'webserver', 'web400', ATTACKER, web, 'GT-1', 'service_scan', 'T1595', 'reconnaissance')
    for i in range(5):
        add(A + 125 + 10 * i, 'webserver', 'web400_multi', ATTACKER, web, 'GT-1', 'service_scan', 'T1595', 'reconnaissance')
    for i in range(100):                                  # 3 wordlist scan (dirb / wpscan)
        add(A + 240 + i, 'webserver', 'new_method', ATTACKER, web, 'GT-1', 'wordlist_scan', 'T1595', 'reconnaissance')
        add(A + 240 + i, 'webserver', 'forbidden', ATTACKER, web, 'GT-1', 'wordlist_scan', 'T1595', 'reconnaissance')
    for i in range(2):                                    # 4 exploit through the WordPress plugin
        add(A + 400 + i, 'webserver', 'web_attack', ATTACKER, web, 'GT-1', 'upload_exploit', 'T1190', 'exploitation')
    add(A + 405, 'webserver', 'suspicious_url', ATTACKER, web, 'GT-1', 'web_shell', 'T1505.003', 'installation')  # 5
    for i in range(5):                                    # 6 commands through the web shell
        add(A + 480 + 40 * i, 'webserver', 'user_cmd', None, None, 'GT-1', 'commands', 'T1059', 'command_and_control')
    add(A + 2400, 'webserver', 'pam_open', None, None, 'GT-1', 'privilege_escalation', 'T1548.003', None)   # 7
    add(A + 2401, 'webserver', 'sudo_first', None, None, 'GT-1', 'privilege_escalation', 'T1548.003', None)
    for i in range(2):
        add(A + 2402 + 10 * i, 'webserver', 'sudo_root', None, None, 'GT-1', 'privilege_escalation', 'T1548.003', None)
    for i in range(40):                                   # 8 data theft over DNS, 6 h later
        add(A + 6 * 3600 + 30 * i, 'inet-dns', 'dns_entropy', web, MACHINE_IP['inet-dns'],
            'GT-2', 'dns_exfiltration', 'T1048', 'actions_on_objectives')
    for i in range(5):
        add(A + 6 * 3600 + 15 + 120 * i, 'inet-dns', 'dns_chars', web, MACHINE_IP['inet-dns'],
            'GT-2', 'dns_exfiltration', 'T1048', 'actions_on_objectives')
    return _finish(rows, 'scenario1', '2026-01-10')


def build_scenario_2():
    """Innocent but noisy. No attacker."""
    rows = []
    add = _adder(rows)
    _background(add)
    for h in range(48):                                   # hourly monitoring scan (same Nmap signature)
        for i in range(10):
            add(h * 3600 + 900 + 2 * i, 'inet-firewall', 'nmap', MACHINE_IP['monitoring'], f'10.0.1.{20 + i}',
                'benign-monitoring', 'monitoring_scan')
    for night in (1, 2):                                  # nightly backup 02:00, 15 min
        for i in range(30):
            add((night - 1) * 86400 + 2 * 3600 + 30 * i, 'internal_share', 'new_event',
                MACHINE_IP['backup'], MACHINE_IP['internal_share'], f'benign-backup-{night}', 'backup')
    D2 = 86400
    for i in range(8):                                    # package updates, then admin sudo
        add(D2 + 11 * 3600 + 15 * i, 'webserver', 'apt', MACHINE_IP['webserver'], '198.51.100.200',
            'benign-update', 'update')
    for i in range(2):
        add(D2 + 11 * 3600 + 150 + 10 * i, 'webserver', 'sudo_root', None, None, 'benign-update', 'admin_sudo')
    for i in range(60):                                   # antivirus full refresh 3 min later
        add(D2 + 11 * 3600 + 180 + i, 'mail', 'clamav', gt='benign-av-refresh', step='av_refresh')
    return _finish(rows, 'scenario2', '2026-01-20')


In [34]:
# ===== S4-6: write scenario files v2 (expectations written BEFORE running) =====
expected = {
    'scenario1': {
        'story': ('Outside attacker 203.0.113.66 scans the network, attacks the WordPress web server, opens a web '
                  'shell, runs commands, escalates to root with sudo, and 6 h later leaks data over DNS. An employee '
                  'mistypes a mail password 5 s after the scan starts.'),
        'ground_truth': {
            'GT-1': {'steps': ['network_scan', 'service_scan', 'wordlist_scan', 'upload_exploit', 'web_shell',
                               'commands', 'privilege_escalation'],
                     'true_techniques': ['T1595', 'T1190', 'T1505.003', 'T1059', 'T1548.003'],
                     'expected_severity': 'High'},
            'GT-2': {'steps': ['dns_exfiltration'], 'true_techniques': ['T1048'], 'expected_severity': 'High',
                     'note': 'same attack story as GT-1'},
            'benign-1': {'what': 'employee mistypes a mail password', 'expected_severity': 'Low'}},
        'expected_agent_v2': {
            'non_routine_incidents': 2,
            'attack': ('ONE incident: firewall scan joined by the shared attacker IP (Rule 3), all web-server steps, '
                       'privilege escalation by a weak link, DNS theft by a weak long-gap link (Rule 5)'),
            'benign-1': 'alone',
            'techniques_all_weak': ['T1048', 'T1059', 'T1190', 'T1548.003', 'T1595'],
            'stages': ['reconnaissance', 'actions_on_objectives'],
            'not_mapped': {'T1505.003': 'no AIT signature specifically shows a web shell'}}},
    'scenario2': {
        'story': ('No attacker. Hourly monitoring scan, nightly backup at 02:00, admin installs updates and uses sudo, '
                  'antivirus full refresh 3 minutes later.'),
        'ground_truth': {'benign-monitoring': {'expected_severity': 'none (routine)'},
                         'benign-backup-1': {'expected_severity': 'Low'}, 'benign-backup-2': {'expected_severity': 'Low'},
                         'benign-update': {'expected_severity': 'Low', 'note': 'weak T1548.003 is expected; must stay Low'},
                         'benign-av-refresh': {'expected_severity': 'Low'}},
        'pass_condition_whole_system': 'no incident rated High or Critical',
        'expected_agent_v2': {'non_routine_incidents': 4, 'update_incident_techniques': ['T1548.003'],
                              'stages': [], 'long_gap_joins': 0}},
}
scenarios = {}
for build in (build_scenario_1, build_scenario_2):
    alerts_s, truth_s = build()
    name = alerts_s['network'].iloc[0]
    scenarios[name] = (alerts_s, truth_s)
    alerts_s.to_csv(SCN / f'{name}_alerts.csv', index=False)
    truth_s.to_csv(SCN / f'{name}_truth.csv', index=False)
    with open(SCN / f'{name}_expected.json', 'w') as f:
        json.dump(expected[name], f, indent=2)
    print(name, len(alerts_s), 'alerts written')


scenario1 813 alerts written
scenario2 1090 alerts written


In [35]:
# ===== S4-7: run agent v2 + Omar's mapper on both scenarios and check the expectations =====
def run_scenario(name):
    alerts_s, truth_s = scenarios[name]
    a, g, links, inc = agent(alerts_s)
    msgs = incident_updates(a, g, links, inc, name)
    with open(Path('out') / f'incident_updates_{name}.jsonl', 'w') as f:
        for m in msgs:
            f.write(json.dumps(m) + chr(10))
    t = a.merge(truth_s, on='alert_id')
    t['kept'] = ~t['incident_id'].isin(set(inc.loc[inc['routine'], 'incident_id']))
    alert_msgs = [alert_message(r) for r in a[a['alert_id'].isin({i for m in msgs for i in m['alert_ids']})].to_dict('records')]
    return t, links, msgs, validate_bundle(alert_msgs + msgs)


Path('out').mkdir(exist_ok=True)
checks = {}
t1, l1, m1, p1 = run_scenario('scenario1')
attack = set(t1.loc[t1['gt_incident'].isin(['GT-1', 'GT-2']), 'incident_id'])
main = next(m for m in m1 if m['incident_id'].replace('inc-scenario1-', 'inc-') in attack)
checks['S1 background all routine'] = not t1.loc[t1['gt_incident'] == 'normal', 'kept'].any()
checks['S1 2 incidents'] = len(m1) == 2
checks['S1 whole attack (GT-1 + GT-2) is ONE incident'] = len(attack) == 1
checks['S1 typo alone'] = set(t1.loc[t1['gt_incident'] == 'benign-1', 'incident_id']).isdisjoint(attack)
checks['S1 DNS theft joined by a weak long-gap link'] = bool(l1['long_gap'].sum() == 1 and (l1.loc[l1['long_gap'], 'strength'] == 'weak').all())
checks['S1 techniques as expected, all weak'] = (sorted(t['technique_id'] for t in main['candidate_techniques']) == expected['scenario1']['expected_agent_v2']['techniques_all_weak']
                                                 and all(t['support'] == 'weak' for t in main['candidate_techniques']))
checks['S1 stages as expected'] = [s['stage'] for s in main['stage_hypotheses']] == expected['scenario1']['expected_agent_v2']['stages']
checks['S1 contract OK'] = p1 == []
t2, l2, m2, p2 = run_scenario('scenario2')
upd = next(m for m in m2 if m['incident_id'].replace('inc-scenario2-', 'inc-') in set(t2.loc[t2['gt_incident'] == 'benign-update', 'incident_id']))
checks['S2 background + monitoring scan all routine'] = not t2.loc[t2['gt_incident'].isin(['normal', 'benign-monitoring']), 'kept'].any()
checks['S2 4 incidents'] = len(m2) == 4
checks['S2 update incident: only weak T1548.003'] = [(t['technique_id'], t['support']) for t in upd['candidate_techniques']] == [('T1548.003', 'weak')]
checks['S2 no stages, no long-gap joins'] = all(not m['stage_hypotheses'] for m in m2) and not l2['long_gap'].any()
checks['S2 no incident mixes two events'] = bool((t2[t2['kept']].groupby('incident_id')['gt_incident'].nunique() == 1).all())
checks['S2 contract OK'] = p2 == []
for name, ok in checks.items():
    print('OK  ' if ok else 'DIFF', name)
print(sum(checks.values()), 'of', len(checks), 'expectations met')
for m in m1 + m2:
    print(m['incident_id'], m['entities']['hosts'], len(m['alert_ids']), 'alerts |', m['confidence'], '|',
          [(t['technique_id'], t['support']) for t in m['candidate_techniques']], [s['stage'] for s in m['stage_hypotheses']])


OK   S1 background all routine
OK   S1 2 incidents
OK   S1 whole attack (GT-1 + GT-2) is ONE incident
OK   S1 typo alone
OK   S1 DNS theft joined by a weak long-gap link
OK   S1 techniques as expected, all weak
OK   S1 stages as expected
OK   S1 contract OK
OK   S2 background + monitoring scan all routine
OK   S2 4 incidents
OK   S2 update incident: only weak T1548.003
OK   S2 no stages, no long-gap joins
OK   S2 no incident mixes two events
OK   S2 contract OK
14 of 14 expectations met
inc-scenario1-000341 ['inet-dns', 'inet-firewall', 'webserver'] 332 alerts | medium | [('T1048', 'weak'), ('T1059', 'weak'), ('T1190', 'weak'), ('T1548.003', 'weak'), ('T1595', 'weak')] ['reconnaissance', 'actions_on_objectives']
inc-scenario1-000343 ['mail'] 1 alerts | low | [] []
inc-scenario2-000023 ['internal_share'] 30 alerts | high | [] []
inc-scenario2-000288 ['internal_share'] 30 alerts | high | [] []
inc-scenario2-000389 ['webserver'] 10 alerts | high | [('T1548.003', 'weak')] []
inc-scenario2-

### Tuning check - v2 must not change the tuned grouping (3 tuning networks only)
Rules 4 and 5 can only weaken links or join incidents across long gaps, so on the tuning networks the attack must still be split into 8 / 7 / 4 with purity 0.982 / 0.994 / 1.000. This cell also writes the IncidentUpdate files to `out/`.

In [36]:
# ===== S4-8: evaluate_network (used for the tuning check now and the final test in Stage 5) =====
def evaluate_network(sc, cfg=CFG):
    """Runs the agent (no labels), writes its IncidentUpdates, then scores against the answers."""
    alerts = pd.read_parquet(OUT / f'{sc}_alerts.parquet')
    a, g, links, inc = agent(alerts, cfg)
    msgs = incident_updates(a, g, links, inc, sc)
    problems = [p for m in msgs for p in validate_message(m)]
    Path('out').mkdir(exist_ok=True)
    with open(Path('out') / f'incident_updates_{sc}.jsonl', 'w') as f:
        for m in msgs:
            f.write(json.dumps(m) + chr(10))

    # ---- EVALUATION: answers are read only from here on ----
    lab = pd.read_parquet(OUT / f'{sc}_eval.parquet')
    p = a[['alert_id', 'incident_id']].merge(lab, on='alert_id')
    p['kept'] = ~p['incident_id'].isin(set(inc.loc[inc['routine'], 'incident_id']))
    p['attack'] = (p['eval_event_label'] != '-') & p['eval_phase'].isin(ATTACK_PHASES)
    normal_time = ~p['eval_phase'].isin(ATTACK_PHASES)
    att = p[p['kept'] & p['attack']]
    counts = att['incident_id'].value_counts()
    top = counts.index[0]
    inside = p[p['incident_id'].isin(counts.index)]
    top_msg = next(m for m in msgs if m['incident_id'] == top.replace('inc-', f'inc-{sc}-', 1))
    in_window = p[p['eval_phase'].isin(ATTACK_PHASES)]
    phases_seen = in_window.groupby('eval_phase')['kept'].any()
    return {
        'network': sc, 'alerts': len(p), 'alerts_kept': round(p['kept'].mean(), 3),
        'incidents': int((~inc['routine']).sum()),
        'attack_kept': round(len(att) / max(int(p['attack'].sum()), 1), 4),
        'false_alarms_kept': round(p.loc[normal_time, 'kept'].mean(), 4),
        'attack_incidents': len(counts),
        'top_share': round(counts.iloc[0] / len(att), 3),
        'top_steps': ', '.join(sorted(set(att.loc[att['incident_id'] == top, 'eval_event_label']))),
        'purity': round(inside['attack'].mean(), 3),
        'phases_seen': f'{int(phases_seen.sum())} of {len(phases_seen)}',
        'top_techniques': ', '.join(t['technique_id'] for t in top_msg['candidate_techniques']),
        'top_stages': ', '.join(s['stage'] for s in top_msg['stage_hypotheses']) or '-',
        'incidents_with_technique': sum(bool(m['candidate_techniques']) for m in msgs),
        'out_of_order_links': int((links['order'] == 'out_of_order').sum()),
        'long_gap_joins': int(links['long_gap'].sum()),
        'contract_problems': len(problems),
    }


TUNE = ['russellmitchell', 'santos', 'shaw']
TEST = ['fox', 'harrison', 'wardbeck', 'wheeler', 'wilson']      # NOT touched until Stage 5
tune_v2 = pd.DataFrame([evaluate_network(sc) for sc in TUNE])
tune_v2.to_csv(Path('out') / 'tuning_v2_results.csv', index=False)
pd.set_option('display.max_colwidth', 120)
print('v1 (Tuning decisions, 23 Nov): attack split into 8 / 7 / 4, purity 0.982 / 0.994 / 1.000')
tune_v2.set_index('network').T


v1 (Tuning decisions, 23 Nov): attack split into 8 / 7 / 4, purity 0.982 / 0.994 / 1.000


network,russellmitchell,santos,shaw
alerts,45544,130779,70782
alerts_kept,0.266,0.095,0.127
incidents,160,183,229
attack_kept,0.9999,0.9999,0.9998
false_alarms_kept,0.0256,0.0097,0.0445
attack_incidents,8,7,4
top_share,0.996,0.998,0.998
top_steps,"attacker_change_user, dirb, escalated_sudo_command, service_scan, webshell_cmd, wpscan","attacker_change_user, dirb, escalated_sudo_command, online_cracking, service_scan, webshell_cmd, wpscan","attacker_change_user, dirb, escalated_sudo_command, webshell_cmd, wpscan"
purity,0.982,0.994,1.0
phases_seen,9 of 10,9 of 10,9 of 10


## Stage 5 - R-6: final evaluation on the 5 test networks (run once)

**Frozen before running:** burst 60 s / cap 10 min, normal day 24 h, routine = 6 slots unless 3x spike, link gap 45 min (strong <= 5 min), Rule 4 order check, Rule 5 long gap <= 24 h, observation table S4-2. Nothing is changed after seeing these results.

**Metrics defined before running (step 5.1)** - "attack" = exact label inside an attack-phase window:
`attack_kept` (share of attack alerts kept), `false_alarms_kept` (share of normal-time alerts kept), `attack_incidents` (how many incidents the attack is split into - lower is better), `top_share` (share of attack alerts in the biggest attack incident), `top_steps`, `purity` (share of alerts inside attack incidents that are attack), `phases_seen`, the main incident's techniques and stages, and `contract_problems` (must be 0).

In [37]:
# ===== S5: R-6 FINAL TEST - the 5 networks never used for design or tuning. Run ONCE. =====
final = pd.DataFrame([evaluate_network(sc) for sc in TEST])
final.to_csv(Path('out') / 'final_test_results.csv', index=False)

num = ['alerts_kept', 'attack_kept', 'false_alarms_kept', 'attack_incidents', 'top_share', 'purity',
       'incidents', 'incidents_with_technique', 'contract_problems']
compare = pd.DataFrame({'tuning (3 networks)': tune_v2[num].mean(),
                        'TEST (5 networks, run once)': final[num].mean()}).round(3)
print(compare.to_string())
print()
final.set_index('network').T


                          tuning (3 networks)  TEST (5 networks, run once)
alerts_kept                             0.163                        0.617
attack_kept                             1.000                        1.000
false_alarms_kept                       0.027                        0.014
attack_incidents                        6.333                        7.000
top_share                               0.997                        0.999
purity                                  0.992                        0.998
incidents                             190.667                      223.800
incidents_with_technique               13.000                       11.200
contract_problems                       0.000                        0.000



network,fox,harrison,wardbeck,wheeler,wilson
alerts,473104,593948,91257,616161,634246
alerts_kept,0.891,0.718,0.088,0.702,0.688
incidents,246,197,245,198,233
attack_kept,1.0,0.9983,0.9998,1.0,1.0
false_alarms_kept,0.0229,0.0075,0.021,0.0083,0.0079
attack_incidents,8,6,5,8,8
top_share,1.0,1.0,0.997,1.0,1.0
top_steps,"attacker_change_user, dirb, escalated_sudo_command, online_cracking, service_scan, webshell_cmd, wpscan","attacker_change_user, dirb, escalated_sudo_command, service_scan, webshell_cmd, wpscan","attacker_change_user, dirb, escalated_sudo_command, service_scan, webshell_cmd, wpscan","dirb, service_scan, webshell_cmd, wpscan","attacker_change_user, dirb, escalated_sudo_command, service_scan, webshell_cmd, wpscan"
purity,1.0,1.0,0.993,1.0,0.999
phases_seen,9 of 10,10 of 10,8 of 10,8 of 9,9 of 10


In [38]:
# ===== S5-view: the same final results as plain text (only displays them; nothing is re-run) =====
cols = ['alerts', 'alerts_kept', 'incidents', 'attack_kept', 'false_alarms_kept', 'attack_incidents', 'top_share',
        'purity', 'phases_seen', 'top_techniques', 'top_stages', 'incidents_with_technique',
        'out_of_order_links', 'long_gap_joins', 'contract_problems']
print(final.set_index('network')[cols].T.to_string())

network                                       fox                        harrison                        wardbeck         wheeler                   wilson
alerts                                     473104                          593948                           91257          616161                   634246
alerts_kept                                 0.891                           0.718                           0.088           0.702                    0.688
incidents                                     246                             197                             245             198                      233
attack_kept                                   1.0                          0.9983                          0.9998             1.0                      1.0
false_alarms_kept                          0.0229                          0.0075                           0.021          0.0083                   0.0079
attack_incidents                                8                     

## R-6 results (24 Nov) - final test on the 5 test networks, run once

| | fox | harrison | wardbeck | wheeler | wilson | **TEST mean** | tuning mean |
|---|---|---|---|---|---|---|---|
| alerts | 473,104 | 593,948 | 91,257 | 616,161 | 634,246 | | |
| attack alerts kept | 1.000 | 0.998 | 1.000 | 1.000 | 1.000 | **1.000** | 1.000 |
| normal-time alerts kept (false alarms) | 0.023 | 0.008 | 0.021 | 0.008 | 0.008 | **0.014** | 0.027 |
| attack split into (incidents) | 8 | 6 | 5 | 8 | 8 | **7.0** | 6.3 |
| share of attack in the biggest incident | 1.000 | 1.000 | 0.997 | 1.000 | 1.000 | **0.999** | 0.997 |
| purity of attack incidents | 1.000 | 1.000 | 0.993 | 1.000 | 0.999 | **0.998** | 0.992 |
| attack phases seen | 9/10 | 10/10 | 8/10 | 8/9 | 9/10 | | |
| main incident techniques (all weak) | T1190, T1548.003, T1595 | T1059, T1190, T1548.003, T1595 | T1059, T1190, T1548.003, T1595 | T1190, T1595 | T1190, T1548.003, T1595 | | |
| main incident stage | reconnaissance | reconnaissance | reconnaissance | reconnaissance | reconnaissance | | |
| contract problems | 0 | 0 | 0 | 0 | 0 | **0** | 0 |

**What this shows.** The settings chosen on the 3 tuning networks carry over to 5 networks the agent never saw: practically every attack alert is kept, 98.6 % of normal-time alerts are removed, the biggest incident holds 99.9 % of the attack, and attack incidents are almost pure. Every IncidentUpdate follows the team contract and cites its evidence.

**Why 'alerts kept' is 0.69-0.89 on four test networks (0.16 on tuning).** Not a fault: on fox, harrison, wheeler and wilson the attacker's `dirb` wordlist scan alone produced about 410,000 alerts, so 69-89 % of *all* alerts there really are attack alerts (checked after the test against the answers). The false-alarm rate (0.8-2.3 %) is the number that matters.

**Limitations (honest).**
- The attack is still split into 5-8 incidents. The biggest holds almost everything; the rest are small pieces (often a step on another machine with no shared IP in the CSV).
- Techniques are **weak** only: IDS signatures alone, no reviewed host evidence. The Response Agent must not treat them as proof.
- Only the reconnaissance stage gets a Kill Chain hypothesis on real data; other steps rarely have two independent alert types in the same incident.
- T1505.003 (web shell) is never mapped: no AIT signature specifically shows a web shell.
- Rule 5 made **0 joins** on real data: the AIT CSV has no src/dst IPs, so incidents on different machines share nothing. It works in Scenario 1, where IPs exist.
- Some innocent incidents get one weak technique (e.g. an admin's sudo -> T1548.003). Expected; they must be rated Low later (Scenario 2 checks this).
- Omar's mapper was changed in 3 small places for this to work; he must review them (`SentinelAI-O-3_v2_for_Omar.zip`).